<a href="https://colab.research.google.com/github/GurnoorKaur-30/internship-project_Gurnoor/blob/main/Eng_Punjabi_DataAligned_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
"""
FULL PIPELINE: English-Punjabi Agromet Bulletin Paragraph-wise Alignment
==========================================================================
Run this as a single script (or paste into one Colab cell). It will:
  1. Unzip your two source zips (if given as zips) or use folders directly.
  2. AUTO-DETECT which side is actually English vs Punjabi by reading a
     sample of the JSON content itself (not the filename/zip label) --
     this prevents the English/Punjabi label-swap issue found earlier.
  3. Align the two languages PARAGRAPH-WISE, section by section, with full
     metadata preserved and ZERO data loss.
  4. Fix the page-break table-splitting issue: a table that spans a page
     break can become 2 separate table-objects in one language's JSON and
     stay 1 object in the other. This script flattens all table-objects in
     a section into one ordered row list, forward-fills blank continuation
     labels, regroups into paragraphs, then matches paragraphs positionally
     -- so a crop's advisory is correctly reunited even if it was split
     differently per language.
  5. Writes one Excel workbook with two sheets:
       - Punjab_Agromet_EN_PA_Aligned  (the aligned paragraph data)
       - JSON_Missing_Values           (bulletins missing on one side entirely)

>>> EDIT ONLY THE "CONFIG" SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# Option A: point directly at two zip files (script will unzip them for you)
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab.zip"

# Option B: if you've already unzipped, set these to the extracted folders instead
# and set ZIP_PATH_1 = ZIP_PATH_2 = None
# FOLDER_1 = "/content/drive/MyDrive/english_extracted"
# FOLDER_2 = "/content/drive/MyDrive/punjabi_extracted"
FOLDER_1 = None
FOLDER_2 = None

EXTRACT_DIR = "/content/pipeline_extracted"             # scratch space for unzipping
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx"

MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# STEP 0: unzip (if zip paths given) and locate the "structured" subfolder
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path

    # find the "structured" folder anywhere under search_root
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


# ============================================================================
# STEP 1: auto-detect which root is English vs Punjabi (by actual script, not filename)
# ============================================================================
GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    """Sample a few JSON files and check if headings are mostly Gurmukhi script or Latin."""
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            heading = sec.get("heading", "")
            if GURMUKHI_RANGE.search(heading):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


# ============================================================================
# STEP 2: document + section lookup
# ============================================================================
def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# STEP 3: table-row flattening (fixes the page-break table-split issue)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    """Flatten all table-objects in a section into one row list, forward-fill
    blank continuation labels, group into paragraph segments. No data is dropped
    -- every row's text is preserved inside some segment."""
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))

    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))

    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])

    return [(label, " ".join(texts).strip()) for label, texts in segments]


# ============================================================================
# STEP 4: main alignment loop
# ============================================================================
def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(
        set(en_docs) | set(pa_docs),
        key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]),
    )

    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))

        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = {s["section_id"]: s for s in (en_doc["sections"] if en_doc else [])}
        pa_sec_map = {s["section_id"]: s for s in (pa_doc["sections"] if pa_doc else [])}

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            records.append({**base, "Content_Type": "heading",
                             "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                             "English_Text": heading_en, "Punjabi_Text": heading_pa})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else ""})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else ""),
                                 "Punjabi_Text": pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")})

            en_segments = flatten_and_group_tables(en_sec)
            pa_segments = flatten_and_group_tables(pa_sec)
            for i in range(max(len(en_segments), len(pa_segments))):
                en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                records.append({**base, "Content_Type": "table_row",
                                 "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                 "English_Text": en_text, "Punjabi_Text": pa_text})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])

    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(
        by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable"
    ).drop(columns="_m").reset_index(drop=True)

    return df, missing_df


# ============================================================================
# STEP 5: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("Step 1/4: Locating 'structured' folders...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")

    print("Step 2/4: Auto-detecting language of each side...")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print("Step 3/4: Aligning paragraph-wise (section-by-section, table-split-safe)...")
    df, missing_df = build_aligned_data(en_root, pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    print("Step 4/4: Writing Excel workbook...")
    write_excel(df, missing_df, OUTPUT_XLSX)
    print(f"Done. Saved to: {OUTPUT_XLSX}")

Step 1/4: Locating 'structured' folders...


FileNotFoundError: [Errno 2] No such file or directory: '/content/Punjab-1.zip'

In [ ]:
!ls -la /content/

total 20
drwxr-xr-x 1 root root 4096 Jul  8 01:32 .
drwxr-xr-x 1 root root 4096 Jul  8 01:30 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:39 .config
drwxr-xr-x 3 root root 4096 Jul  8 01:32 pipeline_extracted
drwxr-xr-x 1 root root 4096 Jun  4 13:39 sample_data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ZIP_PATH_1 = "/content/drive/MyDrive/Punjab_Agromet/Punjab-1.zip"
ZIP_PATH_2 = "/content/drive/MyDrive/Punjab_Agromet/Punjab_1_.zip"

In [ ]:
"""
FULL PIPELINE: English-Punjabi Agromet Bulletin Paragraph-wise Alignment
==========================================================================
Run this as a single script (or paste into one Colab cell). It will:
  1. Unzip your two source zips (if given as zips) or use folders directly.
  2. AUTO-DETECT which side is actually English vs Punjabi by reading a
     sample of the JSON content itself (not the filename/zip label) --
     this prevents the English/Punjabi label-swap issue found earlier.
  3. Align the two languages PARAGRAPH-WISE, section by section, with full
     metadata preserved and ZERO data loss.
  4. Fix the page-break table-splitting issue: a table that spans a page
     break can become 2 separate table-objects in one language's JSON and
     stay 1 object in the other. This script flattens all table-objects in
     a section into one ordered row list, forward-fills blank continuation
     labels, regroups into paragraphs, then matches paragraphs positionally
     -- so a crop's advisory is correctly reunited even if it was split
     differently per language.
  5. Writes one Excel workbook with two sheets:
       - Punjab_Agromet_EN_PA_Aligned  (the aligned paragraph data)
       - JSON_Missing_Values           (bulletins missing on one side entirely)

>>> EDIT ONLY THE "CONFIG" SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# Option A: point directly at two zip files (script will unzip them for you)
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab.zip"

# Option B: if you've already unzipped, set these to the extracted folders instead
# and set ZIP_PATH_1 = ZIP_PATH_2 = None
# FOLDER_1 = "/content/drive/MyDrive/english_extracted"
# FOLDER_2 = "/content/drive/MyDrive/punjabi_extracted"
FOLDER_1 = None
FOLDER_2 = None

EXTRACT_DIR = "/content/pipeline_extracted"             # scratch space for unzipping
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx"

MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# STEP 0: unzip (if zip paths given) and locate the "structured" subfolder
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path

    # find the "structured" folder anywhere under search_root
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


# ============================================================================
# STEP 1: auto-detect which root is English vs Punjabi (by actual script, not filename)
# ============================================================================
GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    """Sample a few JSON files and check if headings are mostly Gurmukhi script or Latin."""
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            heading = sec.get("heading", "")
            if GURMUKHI_RANGE.search(heading):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


# ============================================================================
# STEP 2: document + section lookup
# ============================================================================
def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# STEP 3: table-row flattening (fixes the page-break table-split issue)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    """Flatten all table-objects in a section into one row list, forward-fill
    blank continuation labels, group into paragraph segments. No data is dropped
    -- every row's text is preserved inside some segment."""
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))

    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))

    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])

    return [(label, " ".join(texts).strip()) for label, texts in segments]


# ============================================================================
# STEP 4: main alignment loop
# ============================================================================
def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(
        set(en_docs) | set(pa_docs),
        key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]),
    )

    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))

        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = {s["section_id"]: s for s in (en_doc["sections"] if en_doc else [])}
        pa_sec_map = {s["section_id"]: s for s in (pa_doc["sections"] if pa_doc else [])}

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            records.append({**base, "Content_Type": "heading",
                             "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                             "English_Text": heading_en, "Punjabi_Text": heading_pa})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else ""})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else ""),
                                 "Punjabi_Text": pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")})

            en_segments = flatten_and_group_tables(en_sec)
            pa_segments = flatten_and_group_tables(pa_sec)
            for i in range(max(len(en_segments), len(pa_segments))):
                en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                records.append({**base, "Content_Type": "table_row",
                                 "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                 "English_Text": en_text, "Punjabi_Text": pa_text})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])

    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(
        by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable"
    ).drop(columns="_m").reset_index(drop=True)

    return df, missing_df


# ============================================================================
# STEP 5: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("Step 1/4: Locating 'structured' folders...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")

    print("Step 2/4: Auto-detecting language of each side...")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print("Step 3/4: Aligning paragraph-wise (section-by-section, table-split-safe)...")
    df, missing_df = build_aligned_data(en_root, pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    print("Step 4/4: Writing Excel workbook...")
    write_excel(df, missing_df, OUTPUT_XLSX)
    print(f"Done. Saved to: {OUTPUT_XLSX}")

Step 1/4: Locating 'structured' folders...
Step 2/4: Auto-detecting language of each side...
  -> English content found at: /content/pipeline_extracted/side_2/Punjab/structured
  -> Punjabi content found at: /content/pipeline_extracted/side_1/Punjab/structured
Step 3/4: Aligning paragraph-wise (section-by-section, table-split-safe)...
  -> 27870 aligned rows, 22 bulletins missing on one side
Step 4/4: Writing Excel workbook...
Done. Saved to: /content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx


In [ ]:
"""
FULL PIPELINE: English-Punjabi Agromet Bulletin Paragraph-wise Alignment
==========================================================================
Run this as a single script (or paste into one Colab cell). It will:
  1. Unzip your two source zips (if given as zips) or use folders directly.
  2. AUTO-DETECT which side is actually English vs Punjabi by reading a
     sample of the JSON content itself (not the filename/zip label) --
     this prevents the English/Punjabi label-swap issue found earlier.
  3. Align the two languages PARAGRAPH-WISE, section by section, with full
     metadata preserved and ZERO data loss.
  4. Fix the page-break table-splitting issue: a table that spans a page
     break can become 2 separate table-objects in one language's JSON and
     stay 1 object in the other. This script flattens all table-objects in
     a section into one ordered row list, forward-fills blank continuation
     labels, regroups into paragraphs, then matches paragraphs positionally
     -- so a crop's advisory is correctly reunited even if it was split
     differently per language.
  5. Writes one Excel workbook with two sheets:
       - Punjab_Agromet_EN_PA_Aligned  (the aligned paragraph data)
       - JSON_Missing_Values           (bulletins missing on one side entirely)

>>> EDIT ONLY THE "CONFIG" SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# Option A: point directly at two zip files (script will unzip them for you)
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab.zip"

# Option B: if you've already unzipped, set these to the extracted folders instead
# and set ZIP_PATH_1 = ZIP_PATH_2 = None
# FOLDER_1 = "/content/drive/MyDrive/english_extracted"
# FOLDER_2 = "/content/drive/MyDrive/punjabi_extracted"
FOLDER_1 = None
FOLDER_2 = None

EXTRACT_DIR = "/content/pipeline_extracted"             # scratch space for unzipping
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx"

MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# STEP 0: unzip (if zip paths given) and locate the "structured" subfolder
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path

    # find the "structured" folder anywhere under search_root
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


# ============================================================================
# STEP 1: auto-detect which root is English vs Punjabi (by actual script, not filename)
# ============================================================================
GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    """Sample a few JSON files and check if headings are mostly Gurmukhi script or Latin."""
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            heading = sec.get("heading", "")
            if GURMUKHI_RANGE.search(heading):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


# ============================================================================
# STEP 2: document + section lookup
# ============================================================================
def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# STEP 3: table-row flattening (fixes the page-break table-split issue)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    """Flatten all table-objects in a section into one row list, forward-fill
    blank continuation labels, group into paragraph segments. No data is dropped
    -- every row's text is preserved inside some segment."""
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))

    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))

    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])

    return [(label, " ".join(texts).strip()) for label, texts in segments]


# ============================================================================
# STEP 4: main alignment loop
# ============================================================================
def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(
        set(en_docs) | set(pa_docs),
        key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]),
    )

    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))

        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = {s["section_id"]: s for s in (en_doc["sections"] if en_doc else [])}
        pa_sec_map = {s["section_id"]: s for s in (pa_doc["sections"] if pa_doc else [])}

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            en_present = en_sec is not None
            pa_present = pa_sec is not None
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            def reason_for(en_text, pa_text):
                """Explain WHY a row has a blank side, so it's auditable, not just missing."""
                en_blank = not (en_text and en_text.strip())
                pa_blank = not (pa_text and pa_text.strip())
                if not en_blank and not pa_blank:
                    return "Aligned (both sides present)"
                if en_blank and not pa_blank:
                    if not en_present:
                        return "Whole section missing in English JSON for this date"
                    return "Content gap: this item missing in English JSON (section otherwise present)"
                if pa_blank and not en_blank:
                    if not pa_present:
                        return "Whole section missing in Punjabi JSON for this date"
                    return "Content gap: this item missing in Punjabi JSON (section otherwise present)"
                return "Both sides blank"

            records.append({**base, "Content_Type": "heading",
                             "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                             "English_Text": heading_en, "Punjabi_Text": heading_pa,
                             "Missing_Reason": reason_for(heading_en, heading_pa)})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else "",
                                 "Missing_Reason": reason_for(en_b["text"] if en_b else "", pa_b["text"] if pa_b else "")})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                en_t = en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else "")
                pa_t = pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_t, "Punjabi_Text": pa_t,
                                 "Missing_Reason": reason_for(en_t, pa_t)})

            en_segments = flatten_and_group_tables(en_sec)
            pa_segments = flatten_and_group_tables(pa_sec)
            for i in range(max(len(en_segments), len(pa_segments))):
                en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                records.append({**base, "Content_Type": "table_row",
                                 "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                 "English_Text": en_text, "Punjabi_Text": pa_text,
                                 "Missing_Reason": reason_for(en_text, pa_text)})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text", "Missing_Reason",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])

    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(
        by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable"
    ).drop(columns="_m").reset_index(drop=True)

    return df, missing_df


def build_summary(df):
    """Aggregate missing-content counts by Section_ID and by District, so the
    scale and pattern of gaps is immediately visible and explainable."""
    total = len(df)
    reason_counts = df["Missing_Reason"].value_counts().rename_axis("Missing_Reason").reset_index(name="Row_Count")
    reason_counts["Percent_of_Total"] = (reason_counts["Row_Count"] / total * 100).round(2)

    by_section = (
        df[df["Missing_Reason"] != "Aligned (both sides present)"]
        .groupby(["Section_ID", "Missing_Reason"]).size()
        .reset_index(name="Row_Count")
        .sort_values("Row_Count", ascending=False)
    )

    return reason_counts, by_section


# ============================================================================
# STEP 5: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    # ---- summary sheets: quantify and explain every gap, for review/presentation ----
    reason_counts, by_section = build_summary(df)

    ws3 = wb.create_sheet("Missing_Content_Summary")
    ws3.append(["Overall breakdown: why each row's English_Text / Punjabi_Text is blank (if it is)"])
    ws3["A1"].font = Font(name="Calibri", size=12, bold=True)
    ws3.append([])
    ws3.append(list(reason_counts.columns))
    for cell in ws3[ws3.max_row]:
        cell.font = header_font
    for r in reason_counts.itertuples(index=False):
        ws3.append(list(r))
    for row in ws3.iter_rows(min_row=4, max_row=ws3.max_row):
        for cell in row:
            cell.font = data_font
    ws3.column_dimensions["A"].width = 55
    ws3.column_dimensions["B"].width = 14
    ws3.column_dimensions["C"].width = 16

    start_row = ws3.max_row + 3
    ws3.cell(row=start_row, column=1, value="Breakdown by Section_ID (which sections have the most gaps, and why)").font = Font(name="Calibri", size=12, bold=True)
    header_row = start_row + 2
    for i, h in enumerate(list(by_section.columns), start=1):
        c = ws3.cell(row=header_row, column=i, value=h)
        c.font = header_font
    for r_i, r in enumerate(by_section.itertuples(index=False), start=header_row + 1):
        for c_i, v in enumerate(r, start=1):
            ws3.cell(row=r_i, column=c_i, value=v).font = data_font
    ws3.freeze_panes = "A4"

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("Step 1/4: Locating 'structured' folders...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")

    print("Step 2/4: Auto-detecting language of each side...")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print("Step 3/4: Aligning paragraph-wise (section-by-section, table-split-safe)...")
    df, missing_df = build_aligned_data(en_root, pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    print("Step 4/4: Writing Excel workbook...")
    write_excel(df, missing_df, OUTPUT_XLSX)
    print(f"Done. Saved to: {OUTPUT_XLSX}")

Step 1/4: Locating 'structured' folders...
Step 2/4: Auto-detecting language of each side...
  -> English content found at: /content/pipeline_extracted/side_2/Punjab/structured
  -> Punjabi content found at: /content/pipeline_extracted/side_1/Punjab/structured
Step 3/4: Aligning paragraph-wise (section-by-section, table-split-safe)...
  -> 27870 aligned rows, 22 bulletins missing on one side
Step 4/4: Writing Excel workbook...
Done. Saved to: /content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx


In [ ]:
"""
FULL PIPELINE: English-Punjabi Agromet Bulletin Paragraph-wise Alignment
==========================================================================
Run this as a single script (or paste into one Colab cell). It will:
  1. Unzip your two source zips (if given as zips) or use folders directly.
  2. AUTO-DETECT which side is actually English vs Punjabi by reading a
     sample of the JSON content itself (not the filename/zip label) --
     this prevents the English/Punjabi label-swap issue found earlier.
  3. Align the two languages PARAGRAPH-WISE, section by section, with full
     metadata preserved and ZERO data loss.
  4. Fix the page-break table-splitting issue: a table that spans a page
     break can become 2 separate table-objects in one language's JSON and
     stay 1 object in the other. This script flattens all table-objects in
     a section into one ordered row list, forward-fills blank continuation
     labels, regroups into paragraphs, then matches paragraphs positionally
     -- so a crop's advisory is correctly reunited even if it was split
     differently per language.
  5. Writes one Excel workbook with two sheets:
       - Punjab_Agromet_EN_PA_Aligned  (the aligned paragraph data)
       - JSON_Missing_Values           (bulletins missing on one side entirely)

>>> EDIT ONLY THE "CONFIG" SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# Option A: point directly at two zip files (script will unzip them for you)
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab(1).zip"

# Option B: if you've already unzipped, set these to the extracted folders instead
# and set ZIP_PATH_1 = ZIP_PATH_2 = None
# FOLDER_1 = "/content/drive/MyDrive/english_extracted"
# FOLDER_2 = "/content/drive/MyDrive/punjabi_extracted"
FOLDER_1 = None
FOLDER_2 = None

EXTRACT_DIR = "/content/pipeline_extracted"             # scratch space for unzipping
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx"

MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# STEP 0: unzip (if zip paths given) and locate the "structured" subfolder
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path

    # find the "structured" folder anywhere under search_root
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


# ============================================================================
# STEP 1: auto-detect which root is English vs Punjabi (by actual script, not filename)
# ============================================================================
GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    """Sample a few JSON files and check if headings are mostly Gurmukhi script or Latin."""
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            heading = sec.get("heading", "")
            if GURMUKHI_RANGE.search(heading):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


# ============================================================================
# STEP 2: document + section lookup
# ============================================================================
def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# STEP 3: table-row flattening (fixes the page-break table-split issue)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    """Flatten all table-objects in a section into one row list, forward-fill
    blank continuation labels, group into paragraph segments. No data is dropped
    -- every row's text is preserved inside some segment."""
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))

    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))

    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])

    return [(label, " ".join(texts).strip()) for label, texts in segments]


# ============================================================================
# STEP 4: main alignment loop
# ============================================================================
def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(
        set(en_docs) | set(pa_docs),
        key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]),
    )

    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))

        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = {s["section_id"]: s for s in (en_doc["sections"] if en_doc else [])}
        pa_sec_map = {s["section_id"]: s for s in (pa_doc["sections"] if pa_doc else [])}

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            en_present = en_sec is not None
            pa_present = pa_sec is not None
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            def reason_for(en_text, pa_text):
                """Explain WHY a row has a blank side, so it's auditable, not just missing."""
                en_blank = not (en_text and en_text.strip())
                pa_blank = not (pa_text and pa_text.strip())
                if not en_blank and not pa_blank:
                    return "Aligned (both sides present)"
                if en_blank and not pa_blank:
                    if not en_present:
                        return "Whole section missing in English JSON for this date"
                    return "Content gap: this item missing in English JSON (section otherwise present)"
                if pa_blank and not en_blank:
                    if not pa_present:
                        return "Whole section missing in Punjabi JSON for this date"
                    return "Content gap: this item missing in Punjabi JSON (section otherwise present)"
                return "Both sides blank"

            records.append({**base, "Content_Type": "heading",
                             "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                             "English_Text": heading_en, "Punjabi_Text": heading_pa,
                             "Missing_Reason": reason_for(heading_en, heading_pa)})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else "",
                                 "Missing_Reason": reason_for(en_b["text"] if en_b else "", pa_b["text"] if pa_b else "")})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                en_t = en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else "")
                pa_t = pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_t, "Punjabi_Text": pa_t,
                                 "Missing_Reason": reason_for(en_t, pa_t)})

            en_segments = flatten_and_group_tables(en_sec)
            pa_segments = flatten_and_group_tables(pa_sec)
            for i in range(max(len(en_segments), len(pa_segments))):
                en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                records.append({**base, "Content_Type": "table_row",
                                 "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                 "English_Text": en_text, "Punjabi_Text": pa_text,
                                 "Missing_Reason": reason_for(en_text, pa_text)})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text", "Missing_Reason",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])

    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(
        by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable"
    ).drop(columns="_m").reset_index(drop=True)

    return df, missing_df


def build_summary(df):
    """Aggregate missing-content counts by Section_ID and by District, so the
    scale and pattern of gaps is immediately visible and explainable."""
    total = len(df)
    reason_counts = df["Missing_Reason"].value_counts().rename_axis("Missing_Reason").reset_index(name="Row_Count")
    reason_counts["Percent_of_Total"] = (reason_counts["Row_Count"] / total * 100).round(2)

    by_section = (
        df[df["Missing_Reason"] != "Aligned (both sides present)"]
        .groupby(["Section_ID", "Missing_Reason"]).size()
        .reset_index(name="Row_Count")
        .sort_values("Row_Count", ascending=False)
    )

    return reason_counts, by_section


def extract_tables_only(df):
    """Pull out just the table-row content (crop/horticulture/livestock/weather
    advisory tables) into its own clean, uniformly-structured view -- separated
    from headings and free-text paragraphs so it reads as one aligned table."""
    tables_df = df[df["Content_Type"] == "table_row"].copy()
    tables_df = tables_df.drop(columns=["Content_Type", "Page_No"])
    tables_df = tables_df.rename(columns={
        "Row_Label_ENG": "Crop_Parameter_ENG",
        "Row_Label_PUN": "Crop_Parameter_PUN",
    })
    return tables_df.reset_index(drop=True)


# ============================================================================
# STEP 5: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    # ---- All_Tables_Compiled: every crop/horticulture/livestock/weather table
    # row from both languages, compiled into one uniform, cleanly aligned sheet ----
    tables_df = extract_tables_only(df)
    ws_t = wb.create_sheet("All_Tables_Compiled")
    t_headers = list(tables_df.columns)
    ws_t.append(t_headers)
    for cell in ws_t[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in tables_df.itertuples(index=False):
        ws_t.append(list(row))
    t_wrap_cols = {get_column_letter(t_headers.index(c) + 1) for c in
                   ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text", "Missing_Reason")
                   if c in t_headers}
    for row in ws_t.iter_rows(min_row=2, max_row=ws_t.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in t_wrap_cols \
                else Alignment(vertical="top")
    t_widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
                "Section_Heading_ENG": 26, "Section_Heading_PUN": 26,
                "Crop_Parameter_ENG": 22, "Crop_Parameter_PUN": 22,
                "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(t_headers, start=1):
        ws_t.column_dimensions[get_column_letter(i)].width = t_widths.get(h, 18)
    ws_t.freeze_panes = "A2"
    ws_t.auto_filter.ref = ws_t.dimensions

    # ---- summary sheets: quantify and explain every gap, for review/presentation ----
    reason_counts, by_section = build_summary(df)

    ws3 = wb.create_sheet("Missing_Content_Summary")
    ws3.append(["Overall breakdown: why each row's English_Text / Punjabi_Text is blank (if it is)"])
    ws3["A1"].font = Font(name="Calibri", size=12, bold=True)
    ws3.append([])
    ws3.append(list(reason_counts.columns))
    for cell in ws3[ws3.max_row]:
        cell.font = header_font
    for r in reason_counts.itertuples(index=False):
        ws3.append(list(r))
    for row in ws3.iter_rows(min_row=4, max_row=ws3.max_row):
        for cell in row:
            cell.font = data_font
    ws3.column_dimensions["A"].width = 55
    ws3.column_dimensions["B"].width = 14
    ws3.column_dimensions["C"].width = 16

    start_row = ws3.max_row + 3
    ws3.cell(row=start_row, column=1, value="Breakdown by Section_ID (which sections have the most gaps, and why)").font = Font(name="Calibri", size=12, bold=True)
    header_row = start_row + 2
    for i, h in enumerate(list(by_section.columns), start=1):
        c = ws3.cell(row=header_row, column=i, value=h)
        c.font = header_font
    for r_i, r in enumerate(by_section.itertuples(index=False), start=header_row + 1):
        for c_i, v in enumerate(r, start=1):
            ws3.cell(row=r_i, column=c_i, value=v).font = data_font
    ws3.freeze_panes = "A4"

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("Step 1/4: Locating 'structured' folders...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")

    print("Step 2/4: Auto-detecting language of each side...")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print("Step 3/4: Aligning paragraph-wise (section-by-section, table-split-safe)...")
    df, missing_df = build_aligned_data(en_root, pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    print("Step 4/4: Writing Excel workbook...")
    write_excel(df, missing_df, OUTPUT_XLSX)
    print(f"Done. Saved to: {OUTPUT_XLSX}")

Step 1/4: Locating 'structured' folders...
Step 2/4: Auto-detecting language of each side...
  -> English content found at: /content/pipeline_extracted/side_2/Punjab/structured
  -> Punjabi content found at: /content/pipeline_extracted/side_1/Punjab/structured
Step 3/4: Aligning paragraph-wise (section-by-section, table-split-safe)...
  -> 27870 aligned rows, 22 bulletins missing on one side
Step 4/4: Writing Excel workbook...
Done. Saved to: /content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx


In [ ]:
"""
FULL PIPELINE: English-Punjabi Agromet Bulletin Paragraph-wise Alignment
==========================================================================
Run this as a single script (or paste into one Colab cell). It will:
  1. Unzip your two source zips (if given as zips) or use folders directly.
  2. AUTO-DETECT which side is actually English vs Punjabi by reading a
     sample of the JSON content itself (not the filename/zip label) --
     this prevents the English/Punjabi label-swap issue found earlier.
  3. Align the two languages PARAGRAPH-WISE, section by section, with full
     metadata preserved and ZERO data loss.
  4. Fix the page-break table-splitting issue: a table that spans a page
     break can become 2 separate table-objects in one language's JSON and
     stay 1 object in the other. This script flattens all table-objects in
     a section into one ordered row list, forward-fills blank continuation
     labels, regroups into paragraphs, then matches paragraphs positionally
     -- so a crop's advisory is correctly reunited even if it was split
     differently per language.
  5. Writes one Excel workbook with two sheets:
       - Punjab_Agromet_EN_PA_Aligned  (the aligned paragraph data)
       - JSON_Missing_Values           (bulletins missing on one side entirely)

>>> EDIT ONLY THE "CONFIG" SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# Option A: point directly at two zip files (script will unzip them for you)
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab.zip"

# Option B: if you've already unzipped, set these to the extracted folders instead
# and set ZIP_PATH_1 = ZIP_PATH_2 = None
# FOLDER_1 = "/content/drive/MyDrive/english_extracted"
# FOLDER_2 = "/content/drive/MyDrive/punjabi_extracted"
FOLDER_1 = None
FOLDER_2 = None

EXTRACT_DIR = "/content/pipeline_extracted"             # scratch space for unzipping
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx"

MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# STEP 0: unzip (if zip paths given) and locate the "structured" subfolder
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path

    # find the "structured" folder anywhere under search_root
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


# ============================================================================
# STEP 1: auto-detect which root is English vs Punjabi (by actual script, not filename)
# ============================================================================
GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    """Sample a few JSON files and check if headings are mostly Gurmukhi script or Latin."""
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            heading = sec.get("heading", "")
            if GURMUKHI_RANGE.search(heading):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


# ============================================================================
# STEP 2: document + section lookup
# ============================================================================
def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# STEP 3: table-row flattening (fixes the page-break table-split issue)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    """Flatten all table-objects in a section into one row list, forward-fill
    blank continuation labels, group into paragraph segments. No data is dropped
    -- every row's text is preserved inside some segment."""
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))

    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))

    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])

    return [(label, " ".join(texts).strip()) for label, texts in segments]


# ============================================================================
# STEP 4: main alignment loop
# ============================================================================
def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(
        set(en_docs) | set(pa_docs),
        key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]),
    )

    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))

        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = {s["section_id"]: s for s in (en_doc["sections"] if en_doc else [])}
        pa_sec_map = {s["section_id"]: s for s in (pa_doc["sections"] if pa_doc else [])}

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            en_present = en_sec is not None
            pa_present = pa_sec is not None
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            def reason_for(en_text, pa_text):
                """Explain WHY a row has a blank side, so it's auditable, not just missing."""
                en_blank = not (en_text and en_text.strip())
                pa_blank = not (pa_text and pa_text.strip())
                if not en_blank and not pa_blank:
                    return "Aligned (both sides present)"
                if en_blank and not pa_blank:
                    if not en_present:
                        return "Whole section missing in English JSON for this date"
                    return "Content gap: this item missing in English JSON (section otherwise present)"
                if pa_blank and not en_blank:
                    if not pa_present:
                        return "Whole section missing in Punjabi JSON for this date"
                    return "Content gap: this item missing in Punjabi JSON (section otherwise present)"
                return "Both sides blank"

            records.append({**base, "Content_Type": "heading",
                             "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                             "English_Text": heading_en, "Punjabi_Text": heading_pa,
                             "Missing_Reason": reason_for(heading_en, heading_pa)})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else "",
                                 "Missing_Reason": reason_for(en_b["text"] if en_b else "", pa_b["text"] if pa_b else "")})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                en_t = en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else "")
                pa_t = pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_t, "Punjabi_Text": pa_t,
                                 "Missing_Reason": reason_for(en_t, pa_t)})

            en_segments = flatten_and_group_tables(en_sec)
            pa_segments = flatten_and_group_tables(pa_sec)
            for i in range(max(len(en_segments), len(pa_segments))):
                en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                records.append({**base, "Content_Type": "table_row",
                                 "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                 "English_Text": en_text, "Punjabi_Text": pa_text,
                                 "Missing_Reason": reason_for(en_text, pa_text)})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text", "Missing_Reason",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])

    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(
        by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable"
    ).drop(columns="_m").reset_index(drop=True)

    return df, missing_df


def build_summary(df):
    """Aggregate missing-content counts by Section_ID and by District, so the
    scale and pattern of gaps is immediately visible and explainable."""
    total = len(df)
    reason_counts = df["Missing_Reason"].value_counts().rename_axis("Missing_Reason").reset_index(name="Row_Count")
    reason_counts["Percent_of_Total"] = (reason_counts["Row_Count"] / total * 100).round(2)

    by_section = (
        df[df["Missing_Reason"] != "Aligned (both sides present)"]
        .groupby(["Section_ID", "Missing_Reason"]).size()
        .reset_index(name="Row_Count")
        .sort_values("Row_Count", ascending=False)
    )

    return reason_counts, by_section


def load_all_docs(root):
    """Return {(district, month, date): parsed_json_dict} for every document under root."""
    doc_paths = list_docs(root)
    return {key: load_json(path) for key, path in doc_paths.items()}


DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def is_weather_table(table):
    """The Agromet Advisory weather-forecast table always has 'Parameter' as the
    first column and actual calendar dates as the remaining column headers --
    detect it structurally (language-independent) rather than by section_id."""
    headers = table.get("headers", [])
    if len(headers) < 2:
        return False
    return all(DATE_PATTERN.match(h) for h in headers[1:])

def extract_weather_rows(doc):
    """Scan every section of a document for the weather table and return
    [(parameter_label, [(date, value), ...]), ...] in original row order."""
    if not doc:
        return []
    rows_out = []
    for sec in doc.get("sections", []):
        for t in sec.get("tables", []):
            if not is_weather_table(t):
                continue
            headers = t["headers"]
            param_header = headers[0]
            date_headers = headers[1:]
            for row in t.get("rows", []):
                param = row.get(param_header, "")
                values = [(d, row.get(d, "")) for d in date_headers]
                rows_out.append((param, values))
    return rows_out

def build_weather_df(en_docs_raw, pa_docs_raw, all_keys):
    """Compile the Agromet weather-parameter tables (Wind, Temperature, Rainfall,
    Humidity, Cloud Cover, Warning, etc.) from every district/date into one
    uniform, side-by-side EN/PA sheet. Rows matched positionally by parameter
    order, which is stable across all bulletins (verified against source data)."""
    weather_records = []
    for (district, month, date) in all_keys:
        en_doc = en_docs_raw.get((district, month, date))
        pa_doc = pa_docs_raw.get((district, month, date))
        en_rows = extract_weather_rows(en_doc)
        pa_rows = extract_weather_rows(pa_doc)
        max_rows = max(len(en_rows), len(pa_rows))
        for i in range(max_rows):
            en_param, en_vals = en_rows[i] if i < len(en_rows) else ("", [])
            pa_param, pa_vals = pa_rows[i] if i < len(pa_rows) else ("", [])
            rec = {
                "District": district, "Month": month, "Date": date,
                "Parameter_ENG": en_param, "Parameter_PUN": pa_param,
            }
            missing_bits = []
            if not en_param and pa_param:
                missing_bits.append("Row missing on English side")
            if not pa_param and en_param:
                missing_bits.append("Row missing on Punjabi side")
            for day_idx in range(5):
                en_date, en_val = en_vals[day_idx] if day_idx < len(en_vals) else ("", "")
                pa_date, pa_val = pa_vals[day_idx] if day_idx < len(pa_vals) else ("", "")
                forecast_date = en_date or pa_date
                rec[f"Forecast_Date_Day{day_idx+1}"] = forecast_date
                rec[f"EN_Value_Day{day_idx+1}"] = en_val
                rec[f"PA_Value_Day{day_idx+1}"] = pa_val
                if en_val == "" and pa_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on English side")
                elif pa_val == "" and en_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on Punjabi side")
            rec["Missing_Reason"] = "; ".join(missing_bits) if missing_bits else "Aligned (both sides present)"
            weather_records.append(rec)

    cols = ["District", "Month", "Date", "Parameter_ENG", "Parameter_PUN"]
    for d in range(1, 6):
        cols += [f"Forecast_Date_Day{d}", f"EN_Value_Day{d}", f"PA_Value_Day{d}"]
    cols.append("Missing_Reason")
    wdf = pd.DataFrame(weather_records, columns=cols)
    wdf["_m"] = wdf["Month"].map(MONTH_ORDER).fillna(99)
    wdf = wdf.sort_values(by=["District", "_m", "Date"], kind="stable").drop(columns="_m").reset_index(drop=True)
    return wdf


def extract_tables_only(df):
    """Pull out just the table-row content (crop/horticulture/livestock/weather
    advisory tables) into its own clean, uniformly-structured view -- separated
    from headings and free-text paragraphs so it reads as one aligned table."""
    tables_df = df[df["Content_Type"] == "table_row"].copy()
    tables_df = tables_df.drop(columns=["Content_Type", "Page_No"])
    tables_df = tables_df.rename(columns={
        "Row_Label_ENG": "Crop_Parameter_ENG",
        "Row_Label_PUN": "Crop_Parameter_PUN",
    })
    return tables_df.reset_index(drop=True)


# ============================================================================
# STEP 5: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, weather_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    # ---- All_Tables_Compiled: every crop/horticulture/livestock/weather table
    # row from both languages, compiled into one uniform, cleanly aligned sheet ----
    tables_df = extract_tables_only(df)
    ws_t = wb.create_sheet("All_Tables_Compiled")
    t_headers = list(tables_df.columns)
    ws_t.append(t_headers)
    for cell in ws_t[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in tables_df.itertuples(index=False):
        ws_t.append(list(row))
    t_wrap_cols = {get_column_letter(t_headers.index(c) + 1) for c in
                   ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text", "Missing_Reason")
                   if c in t_headers}
    for row in ws_t.iter_rows(min_row=2, max_row=ws_t.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in t_wrap_cols \
                else Alignment(vertical="top")
    t_widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
                "Section_Heading_ENG": 26, "Section_Heading_PUN": 26,
                "Crop_Parameter_ENG": 22, "Crop_Parameter_PUN": 22,
                "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(t_headers, start=1):
        ws_t.column_dimensions[get_column_letter(i)].width = t_widths.get(h, 18)
    ws_t.freeze_panes = "A2"
    ws_t.auto_filter.ref = ws_t.dimensions

    # ---- Agromet_Weather_Tables: the weather-forecast tables (Wind, Temperature,
    # Rainfall, Humidity, Cloud Cover, Warning) from every district, compiled into
    # one clean, uniformly-columned sheet -- separate from the advisory tables ----
    ws_w = wb.create_sheet("Agromet_Weather_Tables")
    w_headers = list(weather_df.columns)
    ws_w.append(w_headers)
    for cell in ws_w[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in weather_df.itertuples(index=False):
        ws_w.append(list(row))
    w_wrap_cols = {get_column_letter(w_headers.index(c) + 1) for c in
                   ("Parameter_ENG", "Parameter_PUN", "Missing_Reason") if c in w_headers}
    for row in ws_w.iter_rows(min_row=2, max_row=ws_w.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in w_wrap_cols \
                else Alignment(vertical="top")
    w_widths = {"District": 16, "Month": 12, "Date": 12, "Parameter_ENG": 22, "Parameter_PUN": 22,
                "Missing_Reason": 45}
    for i, h in enumerate(w_headers, start=1):
        width = w_widths.get(h, 14 if "Value" in h or "Forecast" in h else 16)
        ws_w.column_dimensions[get_column_letter(i)].width = width
    ws_w.freeze_panes = "A2"
    ws_w.auto_filter.ref = ws_w.dimensions

    # ---- summary sheets: quantify and explain every gap, for review/presentation ----
    reason_counts, by_section = build_summary(df)

    ws3 = wb.create_sheet("Missing_Content_Summary")
    ws3.append(["Overall breakdown: why each row's English_Text / Punjabi_Text is blank (if it is)"])
    ws3["A1"].font = Font(name="Calibri", size=12, bold=True)
    ws3.append([])
    ws3.append(list(reason_counts.columns))
    for cell in ws3[ws3.max_row]:
        cell.font = header_font
    for r in reason_counts.itertuples(index=False):
        ws3.append(list(r))
    for row in ws3.iter_rows(min_row=4, max_row=ws3.max_row):
        for cell in row:
            cell.font = data_font
    ws3.column_dimensions["A"].width = 55
    ws3.column_dimensions["B"].width = 14
    ws3.column_dimensions["C"].width = 16

    start_row = ws3.max_row + 3
    ws3.cell(row=start_row, column=1, value="Breakdown by Section_ID (which sections have the most gaps, and why)").font = Font(name="Calibri", size=12, bold=True)
    header_row = start_row + 2
    for i, h in enumerate(list(by_section.columns), start=1):
        c = ws3.cell(row=header_row, column=i, value=h)
        c.font = header_font
    for r_i, r in enumerate(by_section.itertuples(index=False), start=header_row + 1):
        for c_i, v in enumerate(r, start=1):
            ws3.cell(row=r_i, column=c_i, value=v).font = data_font
    ws3.freeze_panes = "A4"

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("Step 1/4: Locating 'structured' folders...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")

    print("Step 2/4: Auto-detecting language of each side...")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print("Step 3/5: Aligning paragraph-wise (section-by-section, table-split-safe)...")
    df, missing_df = build_aligned_data(en_root, pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    print("Step 4/5: Compiling Agromet weather-parameter tables (Wind, Temperature, Rainfall, etc.)...")
    en_docs_raw = load_all_docs(en_root)
    pa_docs_raw = load_all_docs(pa_root)
    all_keys = sorted(set(en_docs_raw) | set(pa_docs_raw),
                       key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]))
    weather_df = build_weather_df(en_docs_raw, pa_docs_raw, all_keys)
    print(f"  -> {len(weather_df)} weather-parameter rows across all districts")

    print("Step 5/5: Writing Excel workbook...")
    write_excel(df, missing_df, weather_df, OUTPUT_XLSX)
    print(f"Done. Saved to: {OUTPUT_XLSX}")

Step 1/4: Locating 'structured' folders...
Step 2/4: Auto-detecting language of each side...
  -> English content found at: /content/pipeline_extracted/side_2/Punjab/structured
  -> Punjabi content found at: /content/pipeline_extracted/side_1/Punjab/structured
Step 3/5: Aligning paragraph-wise (section-by-section, table-split-safe)...
  -> 27870 aligned rows, 22 bulletins missing on one side
Step 4/5: Compiling Agromet weather-parameter tables (Wind, Temperature, Rainfall, etc.)...
  -> 5751 weather-parameter rows across all districts
Step 5/5: Writing Excel workbook...
Done. Saved to: /content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx


In [ ]:
"""
FULL PIPELINE: English-Punjabi Agromet Bulletin Paragraph-wise Alignment
==========================================================================
Run this as a single script (or paste into one Colab cell). It will:
  1. Unzip your two source zips (if given as zips) or use folders directly.
  2. AUTO-DETECT which side is actually English vs Punjabi by reading a
     sample of the JSON content itself (not the filename/zip label) --
     this prevents the English/Punjabi label-swap issue found earlier.
  3. Align the two languages PARAGRAPH-WISE, section by section, with full
     metadata preserved and ZERO data loss.
  4. Fix the page-break table-splitting issue: a table that spans a page
     break can become 2 separate table-objects in one language's JSON and
     stay 1 object in the other. This script flattens all table-objects in
     a section into one ordered row list, forward-fills blank continuation
     labels, regroups into paragraphs, then matches paragraphs positionally
     -- so a crop's advisory is correctly reunited even if it was split
     differently per language.
  5. Writes one Excel workbook with two sheets:
       - Punjab_Agromet_EN_PA_Aligned  (the aligned paragraph data)
       - JSON_Missing_Values           (bulletins missing on one side entirely)

>>> EDIT ONLY THE "CONFIG" SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# Option A: point directly at two zip files (script will unzip them for you)
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab.zip"

# Option B: if you've already unzipped, set these to the extracted folders instead
# and set ZIP_PATH_1 = ZIP_PATH_2 = None
# FOLDER_1 = "/content/drive/MyDrive/english_extracted"
# FOLDER_2 = "/content/drive/MyDrive/punjabi_extracted"
FOLDER_1 = None
FOLDER_2 = None

EXTRACT_DIR = "/content/pipeline_extracted"             # scratch space for unzipping
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx"

MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# STEP 0: unzip (if zip paths given) and locate the "structured" subfolder
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path

    # find the "structured" folder anywhere under search_root
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


# ============================================================================
# STEP 1: auto-detect which root is English vs Punjabi (by actual script, not filename)
# ============================================================================
GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    """Sample a few JSON files and check if headings are mostly Gurmukhi script or Latin."""
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            heading = sec.get("heading", "")
            if GURMUKHI_RANGE.search(heading):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


# ============================================================================
# STEP 2: document + section lookup
# ============================================================================
def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# STEP 3: table-row flattening (fixes the page-break table-split issue)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    """Flatten all table-objects in a section into one row list, forward-fill
    blank continuation labels, group into paragraph segments. No data is dropped
    -- every row's text is preserved inside some segment."""
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))

    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))

    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])

    return [(label, " ".join(texts).strip()) for label, texts in segments]


# ============================================================================
# STEP 4: main alignment loop
# ============================================================================
def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(
        set(en_docs) | set(pa_docs),
        key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]),
    )

    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))

        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = {s["section_id"]: s for s in (en_doc["sections"] if en_doc else [])}
        pa_sec_map = {s["section_id"]: s for s in (pa_doc["sections"] if pa_doc else [])}

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            en_present = en_sec is not None
            pa_present = pa_sec is not None
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            def reason_for(en_text, pa_text):
                """Explain WHY a row has a blank side, so it's auditable, not just missing."""
                en_blank = not (en_text and en_text.strip())
                pa_blank = not (pa_text and pa_text.strip())
                if not en_blank and not pa_blank:
                    return "Aligned (both sides present)"
                if en_blank and not pa_blank:
                    if not en_present:
                        return "Whole section missing in English JSON for this date"
                    return "Content gap: this item missing in English JSON (section otherwise present)"
                if pa_blank and not en_blank:
                    if not pa_present:
                        return "Whole section missing in Punjabi JSON for this date"
                    return "Content gap: this item missing in Punjabi JSON (section otherwise present)"
                return "Both sides blank"

            records.append({**base, "Content_Type": "heading",
                             "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                             "English_Text": heading_en, "Punjabi_Text": heading_pa,
                             "Missing_Reason": reason_for(heading_en, heading_pa)})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else "",
                                 "Missing_Reason": reason_for(en_b["text"] if en_b else "", pa_b["text"] if pa_b else "")})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                en_t = en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else "")
                pa_t = pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_t, "Punjabi_Text": pa_t,
                                 "Missing_Reason": reason_for(en_t, pa_t)})

            en_segments = flatten_and_group_tables(en_sec)
            pa_segments = flatten_and_group_tables(pa_sec)
            for i in range(max(len(en_segments), len(pa_segments))):
                en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                records.append({**base, "Content_Type": "table_row",
                                 "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                 "English_Text": en_text, "Punjabi_Text": pa_text,
                                 "Missing_Reason": reason_for(en_text, pa_text)})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text", "Missing_Reason",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])

    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(
        by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable"
    ).drop(columns="_m").reset_index(drop=True)

    return df, missing_df


def build_summary(df):
    """Aggregate missing-content counts by Section_ID and by District, so the
    scale and pattern of gaps is immediately visible and explainable."""
    total = len(df)
    reason_counts = df["Missing_Reason"].value_counts().rename_axis("Missing_Reason").reset_index(name="Row_Count")
    reason_counts["Percent_of_Total"] = (reason_counts["Row_Count"] / total * 100).round(2)

    by_section = (
        df[df["Missing_Reason"] != "Aligned (both sides present)"]
        .groupby(["Section_ID", "Missing_Reason"]).size()
        .reset_index(name="Row_Count")
        .sort_values("Row_Count", ascending=False)
    )

    return reason_counts, by_section


def load_all_docs(root):
    """Return {(district, month, date): parsed_json_dict} for every document under root."""
    doc_paths = list_docs(root)
    return {key: load_json(path) for key, path in doc_paths.items()}


DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def is_weather_table(table):
    """The Agromet Advisory weather-forecast table always has 'Parameter' as the
    first column and actual calendar dates as the remaining column headers --
    detect it structurally (language-independent) rather than by section_id."""
    headers = table.get("headers", [])
    if len(headers) < 2:
        return False
    return all(DATE_PATTERN.match(h) for h in headers[1:])

def extract_weather_rows(doc):
    """Scan every section of a document for the weather table and return
    [(parameter_label, [(date, value), ...]), ...] in original row order."""
    if not doc:
        return []
    rows_out = []
    for sec in doc.get("sections", []):
        for t in sec.get("tables", []):
            if not is_weather_table(t):
                continue
            headers = t["headers"]
            param_header = headers[0]
            date_headers = headers[1:]
            for row in t.get("rows", []):
                param = row.get(param_header, "")
                values = [(d, row.get(d, "")) for d in date_headers]
                rows_out.append((param, values))
    return rows_out

def build_weather_df(en_docs_raw, pa_docs_raw, all_keys):
    """Compile the Agromet weather-parameter tables (Wind, Temperature, Rainfall,
    Humidity, Cloud Cover, Warning, etc.) from every district/date into one
    uniform, side-by-side EN/PA sheet. Rows matched positionally by parameter
    order, which is stable across all bulletins (verified against source data)."""
    weather_records = []
    for (district, month, date) in all_keys:
        en_doc = en_docs_raw.get((district, month, date))
        pa_doc = pa_docs_raw.get((district, month, date))
        en_rows = extract_weather_rows(en_doc)
        pa_rows = extract_weather_rows(pa_doc)
        max_rows = max(len(en_rows), len(pa_rows))
        for i in range(max_rows):
            en_param, en_vals = en_rows[i] if i < len(en_rows) else ("", [])
            pa_param, pa_vals = pa_rows[i] if i < len(pa_rows) else ("", [])
            rec = {
                "District": district, "Month": month, "Date": date,
                "Parameter_ENG": en_param, "Parameter_PUN": pa_param,
            }
            missing_bits = []
            if not en_param and pa_param:
                missing_bits.append("Row missing on English side")
            if not pa_param and en_param:
                missing_bits.append("Row missing on Punjabi side")
            for day_idx in range(5):
                en_date, en_val = en_vals[day_idx] if day_idx < len(en_vals) else ("", "")
                pa_date, pa_val = pa_vals[day_idx] if day_idx < len(pa_vals) else ("", "")
                forecast_date = en_date or pa_date
                rec[f"Forecast_Date_Day{day_idx+1}"] = forecast_date
                rec[f"EN_Value_Day{day_idx+1}"] = en_val
                rec[f"PA_Value_Day{day_idx+1}"] = pa_val
                if en_val == "" and pa_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on English side")
                elif pa_val == "" and en_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on Punjabi side")
            rec["Missing_Reason"] = "; ".join(missing_bits) if missing_bits else "Aligned (both sides present)"
            weather_records.append(rec)

    cols = ["District", "Month", "Date", "Parameter_ENG", "Parameter_PUN"]
    for d in range(1, 6):
        cols += [f"Forecast_Date_Day{d}", f"EN_Value_Day{d}", f"PA_Value_Day{d}"]
    cols.append("Missing_Reason")
    wdf = pd.DataFrame(weather_records, columns=cols)
    wdf["_m"] = wdf["Month"].map(MONTH_ORDER).fillna(99)
    wdf = wdf.sort_values(by=["District", "_m", "Date"], kind="stable").drop(columns="_m").reset_index(drop=True)
    return wdf






# ============================================================================
# STEP 5: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, weather_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    # ---- Agromet_Weather_Tables sheet is written below ----

    # ---- Agromet_Weather_Tables: the weather-forecast tables (Wind, Temperature,
    # Rainfall, Humidity, Cloud Cover, Warning) from every district, compiled into
    # one clean, uniformly-columned sheet -- separate from the advisory tables ----
    ws_w = wb.create_sheet("Agromet_Weather_Tables")
    w_headers = list(weather_df.columns)
    ws_w.append(w_headers)
    for cell in ws_w[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in weather_df.itertuples(index=False):
        ws_w.append(list(row))
    w_wrap_cols = {get_column_letter(w_headers.index(c) + 1) for c in
                   ("Parameter_ENG", "Parameter_PUN", "Missing_Reason") if c in w_headers}
    for row in ws_w.iter_rows(min_row=2, max_row=ws_w.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in w_wrap_cols \
                else Alignment(vertical="top")
    w_widths = {"District": 16, "Month": 12, "Date": 12, "Parameter_ENG": 22, "Parameter_PUN": 22,
                "Missing_Reason": 45}
    for i, h in enumerate(w_headers, start=1):
        width = w_widths.get(h, 14 if "Value" in h or "Forecast" in h else 16)
        ws_w.column_dimensions[get_column_letter(i)].width = width
    ws_w.freeze_panes = "A2"
    ws_w.auto_filter.ref = ws_w.dimensions

    # ---- summary sheets: quantify and explain every gap, for review/presentation ----
    reason_counts, by_section = build_summary(df)

    ws3 = wb.create_sheet("Missing_Content_Summary")
    ws3.append(["Overall breakdown: why each row's English_Text / Punjabi_Text is blank (if it is)"])
    ws3["A1"].font = Font(name="Calibri", size=12, bold=True)
    ws3.append([])
    ws3.append(list(reason_counts.columns))
    for cell in ws3[ws3.max_row]:
        cell.font = header_font
    for r in reason_counts.itertuples(index=False):
        ws3.append(list(r))
    for row in ws3.iter_rows(min_row=4, max_row=ws3.max_row):
        for cell in row:
            cell.font = data_font
    ws3.column_dimensions["A"].width = 55
    ws3.column_dimensions["B"].width = 14
    ws3.column_dimensions["C"].width = 16

    start_row = ws3.max_row + 3
    ws3.cell(row=start_row, column=1, value="Breakdown by Section_ID (which sections have the most gaps, and why)").font = Font(name="Calibri", size=12, bold=True)
    header_row = start_row + 2
    for i, h in enumerate(list(by_section.columns), start=1):
        c = ws3.cell(row=header_row, column=i, value=h)
        c.font = header_font
    for r_i, r in enumerate(by_section.itertuples(index=False), start=header_row + 1):
        for c_i, v in enumerate(r, start=1):
            ws3.cell(row=r_i, column=c_i, value=v).font = data_font
    ws3.freeze_panes = "A4"

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("Step 1/4: Locating 'structured' folders...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")

    print("Step 2/4: Auto-detecting language of each side...")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print("Step 3/5: Aligning paragraph-wise (section-by-section, table-split-safe)...")
    df, missing_df = build_aligned_data(en_root, pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    print("Step 4/5: Compiling Agromet weather-parameter tables (Wind, Temperature, Rainfall, etc.)...")
    en_docs_raw = load_all_docs(en_root)
    pa_docs_raw = load_all_docs(pa_root)
    all_keys = sorted(set(en_docs_raw) | set(pa_docs_raw),
                       key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]))
    weather_df = build_weather_df(en_docs_raw, pa_docs_raw, all_keys)
    print(f"  -> {len(weather_df)} weather-parameter rows across all districts")

    print("Step 5/5: Writing Excel workbook...")
    write_excel(df, missing_df, weather_df, OUTPUT_XLSX)
    print(f"Done. Saved to: {OUTPUT_XLSX}")

Step 1/4: Locating 'structured' folders...


FileNotFoundError: [Errno 2] No such file or directory: '/content/Punjab-1.zip'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ZIP_PATH_1 = "/content/drive/MyDrive/Punjab_Agromet/Punjab(1).zip"
ZIP_PATH_2 = "/content/drive/MyDrive/Punjab_Agromet/Punjab-1.zip"

In [ ]:
"""
FULL PIPELINE: English-Punjabi Agromet Bulletin Paragraph-wise Alignment
==========================================================================
Run this as a single script (or paste into one Colab cell). It will:
  1. Unzip your two source zips (if given as zips) or use folders directly.
  2. AUTO-DETECT which side is actually English vs Punjabi by reading a
     sample of the JSON content itself (not the filename/zip label) --
     this prevents the English/Punjabi label-swap issue found earlier.
  3. Align the two languages PARAGRAPH-WISE, section by section, with full
     metadata preserved and ZERO data loss.
  4. Fix the page-break table-splitting issue: a table that spans a page
     break can become 2 separate table-objects in one language's JSON and
     stay 1 object in the other. This script flattens all table-objects in
     a section into one ordered row list, forward-fills blank continuation
     labels, regroups into paragraphs, then matches paragraphs positionally
     -- so a crop's advisory is correctly reunited even if it was split
     differently per language.
  5. Writes one Excel workbook with two sheets:
       - Punjab_Agromet_EN_PA_Aligned  (the aligned paragraph data)
       - JSON_Missing_Values           (bulletins missing on one side entirely)

>>> EDIT ONLY THE "CONFIG" SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# Option A: point directly at two zip files (script will unzip them for you)
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab(1).zip"

# Option B: if you've already unzipped, set these to the extracted folders instead
# and set ZIP_PATH_1 = ZIP_PATH_2 = None
# FOLDER_1 = "/content/drive/MyDrive/english_extracted"
# FOLDER_2 = "/content/drive/MyDrive/punjabi_extracted"
FOLDER_1 = None
FOLDER_2 = None

EXTRACT_DIR = "/content/pipeline_extracted"             # scratch space for unzipping
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx"

MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# STEP 0: unzip (if zip paths given) and locate the "structured" subfolder
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path

    # find the "structured" folder anywhere under search_root
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


# ============================================================================
# STEP 1: auto-detect which root is English vs Punjabi (by actual script, not filename)
# ============================================================================
GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    """Sample a few JSON files and check if headings are mostly Gurmukhi script or Latin."""
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            heading = sec.get("heading", "")
            if GURMUKHI_RANGE.search(heading):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


# ============================================================================
# STEP 2: document + section lookup
# ============================================================================
def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# STEP 3: table-row flattening (fixes the page-break table-split issue)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    """Flatten all table-objects in a section into one row list, forward-fill
    blank continuation labels, group into paragraph segments. No data is dropped
    -- every row's text is preserved inside some segment."""
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))

    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))

    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])

    return [(label, " ".join(texts).strip()) for label, texts in segments]


# ============================================================================
# STEP 4: main alignment loop
# ============================================================================
def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(
        set(en_docs) | set(pa_docs),
        key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]),
    )

    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))

        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = {s["section_id"]: s for s in (en_doc["sections"] if en_doc else [])}
        pa_sec_map = {s["section_id"]: s for s in (pa_doc["sections"] if pa_doc else [])}

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            en_present = en_sec is not None
            pa_present = pa_sec is not None
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            def reason_for(en_text, pa_text):
                """Explain WHY a row has a blank side, so it's auditable, not just missing."""
                en_blank = not (en_text and en_text.strip())
                pa_blank = not (pa_text and pa_text.strip())
                if not en_blank and not pa_blank:
                    return "Aligned (both sides present)"
                if en_blank and not pa_blank:
                    if not en_present:
                        return "Whole section missing in English JSON for this date"
                    return "Content gap: this item missing in English JSON (section otherwise present)"
                if pa_blank and not en_blank:
                    if not pa_present:
                        return "Whole section missing in Punjabi JSON for this date"
                    return "Content gap: this item missing in Punjabi JSON (section otherwise present)"
                return "Both sides blank"

            records.append({**base, "Content_Type": "heading",
                             "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                             "English_Text": heading_en, "Punjabi_Text": heading_pa,
                             "Missing_Reason": reason_for(heading_en, heading_pa)})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else "",
                                 "Missing_Reason": reason_for(en_b["text"] if en_b else "", pa_b["text"] if pa_b else "")})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                en_t = en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else "")
                pa_t = pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_t, "Punjabi_Text": pa_t,
                                 "Missing_Reason": reason_for(en_t, pa_t)})

            en_segments = flatten_and_group_tables(en_sec)
            pa_segments = flatten_and_group_tables(pa_sec)
            for i in range(max(len(en_segments), len(pa_segments))):
                en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                records.append({**base, "Content_Type": "table_row",
                                 "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                 "English_Text": en_text, "Punjabi_Text": pa_text,
                                 "Missing_Reason": reason_for(en_text, pa_text)})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text", "Missing_Reason",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])

    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(
        by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable"
    ).drop(columns="_m").reset_index(drop=True)

    return df, missing_df


def build_summary(df):
    """Aggregate missing-content counts by Section_ID and by District, so the
    scale and pattern of gaps is immediately visible and explainable."""
    total = len(df)
    reason_counts = df["Missing_Reason"].value_counts().rename_axis("Missing_Reason").reset_index(name="Row_Count")
    reason_counts["Percent_of_Total"] = (reason_counts["Row_Count"] / total * 100).round(2)

    by_section = (
        df[df["Missing_Reason"] != "Aligned (both sides present)"]
        .groupby(["Section_ID", "Missing_Reason"]).size()
        .reset_index(name="Row_Count")
        .sort_values("Row_Count", ascending=False)
    )

    return reason_counts, by_section


def load_all_docs(root):
    """Return {(district, month, date): parsed_json_dict} for every document under root."""
    doc_paths = list_docs(root)
    return {key: load_json(path) for key, path in doc_paths.items()}


DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def is_weather_table(table):
    """The Agromet Advisory weather-forecast table always has 'Parameter' as the
    first column and actual calendar dates as the remaining column headers --
    detect it structurally (language-independent) rather than by section_id."""
    headers = table.get("headers", [])
    if len(headers) < 2:
        return False
    return all(DATE_PATTERN.match(h) for h in headers[1:])

def extract_weather_rows(doc):
    """Scan every section of a document for the weather table and return
    [(parameter_label, [(date, value), ...]), ...] in original row order."""
    if not doc:
        return []
    rows_out = []
    for sec in doc.get("sections", []):
        for t in sec.get("tables", []):
            if not is_weather_table(t):
                continue
            headers = t["headers"]
            param_header = headers[0]
            date_headers = headers[1:]
            for row in t.get("rows", []):
                param = row.get(param_header, "")
                values = [(d, row.get(d, "")) for d in date_headers]
                rows_out.append((param, values))
    return rows_out

def build_weather_df(en_docs_raw, pa_docs_raw, all_keys):
    """Compile the Agromet weather-parameter tables (Wind, Temperature, Rainfall,
    Humidity, Cloud Cover, Warning, etc.) from every district/date into one
    uniform, side-by-side EN/PA sheet. Rows matched positionally by parameter
    order, which is stable across all bulletins (verified against source data)."""
    weather_records = []
    for (district, month, date) in all_keys:
        en_doc = en_docs_raw.get((district, month, date))
        pa_doc = pa_docs_raw.get((district, month, date))
        en_rows = extract_weather_rows(en_doc)
        pa_rows = extract_weather_rows(pa_doc)
        max_rows = max(len(en_rows), len(pa_rows))
        for i in range(max_rows):
            en_param, en_vals = en_rows[i] if i < len(en_rows) else ("", [])
            pa_param, pa_vals = pa_rows[i] if i < len(pa_rows) else ("", [])
            rec = {
                "District": district, "Month": month, "Date": date,
                "Parameter_ENG": en_param, "Parameter_PUN": pa_param,
            }
            missing_bits = []
            if not en_param and pa_param:
                missing_bits.append("Row missing on English side")
            if not pa_param and en_param:
                missing_bits.append("Row missing on Punjabi side")
            for day_idx in range(5):
                en_date, en_val = en_vals[day_idx] if day_idx < len(en_vals) else ("", "")
                pa_date, pa_val = pa_vals[day_idx] if day_idx < len(pa_vals) else ("", "")
                forecast_date = en_date or pa_date
                rec[f"Forecast_Date_Day{day_idx+1}"] = forecast_date
                rec[f"EN_Value_Day{day_idx+1}"] = en_val
                rec[f"PA_Value_Day{day_idx+1}"] = pa_val
                if en_val == "" and pa_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on English side")
                elif pa_val == "" and en_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on Punjabi side")
            rec["Missing_Reason"] = "; ".join(missing_bits) if missing_bits else "Aligned (both sides present)"
            weather_records.append(rec)

    cols = ["District", "Month", "Date", "Parameter_ENG", "Parameter_PUN"]
    for d in range(1, 6):
        cols += [f"Forecast_Date_Day{d}", f"EN_Value_Day{d}", f"PA_Value_Day{d}"]
    cols.append("Missing_Reason")
    wdf = pd.DataFrame(weather_records, columns=cols)
    wdf["_m"] = wdf["Month"].map(MONTH_ORDER).fillna(99)
    wdf = wdf.sort_values(by=["District", "_m", "Date"], kind="stable").drop(columns="_m").reset_index(drop=True)
    return wdf


def extract_tables_only(df):
    """Pull out just the table-row content (crop/horticulture/livestock/weather
    advisory tables) into its own clean, uniformly-structured view -- separated
    from headings and free-text paragraphs so it reads as one aligned table."""
    tables_df = df[df["Content_Type"] == "table_row"].copy()
    tables_df = tables_df.drop(columns=["Content_Type", "Page_No"])
    tables_df = tables_df.rename(columns={
        "Row_Label_ENG": "Crop_Parameter_ENG",
        "Row_Label_PUN": "Crop_Parameter_PUN",
    })
    return tables_df.reset_index(drop=True)


# ============================================================================
# STEP 5: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, weather_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    # ---- All_Tables_Compiled: every crop/horticulture/livestock/weather table
    # row from both languages, compiled into one uniform, cleanly aligned sheet ----
    tables_df = extract_tables_only(df)
    ws_t = wb.create_sheet("All_Tables_Compiled")
    t_headers = list(tables_df.columns)
    ws_t.append(t_headers)
    for cell in ws_t[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in tables_df.itertuples(index=False):
        ws_t.append(list(row))
    t_wrap_cols = {get_column_letter(t_headers.index(c) + 1) for c in
                   ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text", "Missing_Reason")
                   if c in t_headers}
    for row in ws_t.iter_rows(min_row=2, max_row=ws_t.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in t_wrap_cols \
                else Alignment(vertical="top")
    t_widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
                "Section_Heading_ENG": 26, "Section_Heading_PUN": 26,
                "Crop_Parameter_ENG": 22, "Crop_Parameter_PUN": 22,
                "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(t_headers, start=1):
        ws_t.column_dimensions[get_column_letter(i)].width = t_widths.get(h, 18)
    ws_t.freeze_panes = "A2"
    ws_t.auto_filter.ref = ws_t.dimensions

    # ---- Agromet_Weather_Tables: the weather-forecast tables (Wind, Temperature,
    # Rainfall, Humidity, Cloud Cover, Warning) from every district, compiled into
    # one clean, uniformly-columned sheet -- separate from the advisory tables ----
    ws_w = wb.create_sheet("Agromet_Weather_Tables")
    w_headers = list(weather_df.columns)
    ws_w.append(w_headers)
    for cell in ws_w[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in weather_df.itertuples(index=False):
        ws_w.append(list(row))
    w_wrap_cols = {get_column_letter(w_headers.index(c) + 1) for c in
                   ("Parameter_ENG", "Parameter_PUN", "Missing_Reason") if c in w_headers}
    for row in ws_w.iter_rows(min_row=2, max_row=ws_w.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in w_wrap_cols \
                else Alignment(vertical="top")
    w_widths = {"District": 16, "Month": 12, "Date": 12, "Parameter_ENG": 22, "Parameter_PUN": 22,
                "Missing_Reason": 45}
    for i, h in enumerate(w_headers, start=1):
        width = w_widths.get(h, 14 if "Value" in h or "Forecast" in h else 16)
        ws_w.column_dimensions[get_column_letter(i)].width = width
    ws_w.freeze_panes = "A2"
    ws_w.auto_filter.ref = ws_w.dimensions

    # ---- summary sheets: quantify and explain every gap, for review/presentation ----
    reason_counts, by_section = build_summary(df)

    ws3 = wb.create_sheet("Missing_Content_Summary")
    ws3.append(["Overall breakdown: why each row's English_Text / Punjabi_Text is blank (if it is)"])
    ws3["A1"].font = Font(name="Calibri", size=12, bold=True)
    ws3.append([])
    ws3.append(list(reason_counts.columns))
    for cell in ws3[ws3.max_row]:
        cell.font = header_font
    for r in reason_counts.itertuples(index=False):
        ws3.append(list(r))
    for row in ws3.iter_rows(min_row=4, max_row=ws3.max_row):
        for cell in row:
            cell.font = data_font
    ws3.column_dimensions["A"].width = 55
    ws3.column_dimensions["B"].width = 14
    ws3.column_dimensions["C"].width = 16

    start_row = ws3.max_row + 3
    ws3.cell(row=start_row, column=1, value="Breakdown by Section_ID (which sections have the most gaps, and why)").font = Font(name="Calibri", size=12, bold=True)
    header_row = start_row + 2
    for i, h in enumerate(list(by_section.columns), start=1):
        c = ws3.cell(row=header_row, column=i, value=h)
        c.font = header_font
    for r_i, r in enumerate(by_section.itertuples(index=False), start=header_row + 1):
        for c_i, v in enumerate(r, start=1):
            ws3.cell(row=r_i, column=c_i, value=v).font = data_font
    ws3.freeze_panes = "A4"

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("Step 1/4: Locating 'structured' folders...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")

    print("Step 2/4: Auto-detecting language of each side...")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print("Step 3/5: Aligning paragraph-wise (section-by-section, table-split-safe)...")
    df, missing_df = build_aligned_data(en_root, pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    print("Step 4/5: Compiling Agromet weather-parameter tables (Wind, Temperature, Rainfall, etc.)...")
    en_docs_raw = load_all_docs(en_root)
    pa_docs_raw = load_all_docs(pa_root)
    all_keys = sorted(set(en_docs_raw) | set(pa_docs_raw),
                       key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]))
    weather_df = build_weather_df(en_docs_raw, pa_docs_raw, all_keys)
    print(f"  -> {len(weather_df)} weather-parameter rows across all districts")

    print("Step 5/5: Writing Excel workbook...")
    write_excel(df, missing_df, weather_df, OUTPUT_XLSX)
    print(f"Done. Saved to: {OUTPUT_XLSX}")

Step 1/4: Locating 'structured' folders...
Step 2/4: Auto-detecting language of each side...
  -> English content found at: /content/pipeline_extracted/side_2/Punjab/structured
  -> Punjabi content found at: /content/pipeline_extracted/side_1/Punjab/structured
Step 3/5: Aligning paragraph-wise (section-by-section, table-split-safe)...
  -> 27870 aligned rows, 22 bulletins missing on one side
Step 4/5: Compiling Agromet weather-parameter tables (Wind, Temperature, Rainfall, etc.)...
  -> 5751 weather-parameter rows across all districts
Step 5/5: Writing Excel workbook...
Done. Saved to: /content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx


In [ ]:
"""
FULL PIPELINE: English-Punjabi Agromet Bulletin Paragraph-wise Alignment
==========================================================================
Run this as a single script (or paste into one Colab cell). It will:
  1. Unzip your two source zips (if given as zips) or use folders directly.
  2. AUTO-DETECT which side is actually English vs Punjabi by reading a
     sample of the JSON content itself (not the filename/zip label) --
     this prevents the English/Punjabi label-swap issue found earlier.
  3. Align the two languages PARAGRAPH-WISE, section by section, with full
     metadata preserved and ZERO data loss.
  4. Fix the page-break table-splitting issue: a table that spans a page
     break can become 2 separate table-objects in one language's JSON and
     stay 1 object in the other. This script flattens all table-objects in
     a section into one ordered row list, forward-fills blank continuation
     labels, regroups into paragraphs, then matches paragraphs positionally
     -- so a crop's advisory is correctly reunited even if it was split
     differently per language.
  5. Writes one Excel workbook with two sheets:
       - Punjab_Agromet_EN_PA_Aligned  (the aligned paragraph data)
       - JSON_Missing_Values           (bulletins missing on one side entirely)

>>> EDIT ONLY THE "CONFIG" SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# Option A: point directly at two zip files (script will unzip them for you)
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab(1).zip"

# Option B: if you've already unzipped, set these to the extracted folders instead
# and set ZIP_PATH_1 = ZIP_PATH_2 = None
# FOLDER_1 = "/content/drive/MyDrive/english_extracted"
# FOLDER_2 = "/content/drive/MyDrive/punjabi_extracted"
FOLDER_1 = None
FOLDER_2 = None

EXTRACT_DIR = "/content/pipeline_extracted"             # scratch space for unzipping
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx"

MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# STEP 0: unzip (if zip paths given) and locate the "structured" subfolder
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path

    # find the "structured" folder anywhere under search_root
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


# ============================================================================
# STEP 1: auto-detect which root is English vs Punjabi (by actual script, not filename)
# ============================================================================
GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    """Sample a few JSON files and check if headings are mostly Gurmukhi script or Latin."""
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            heading = sec.get("heading", "")
            if GURMUKHI_RANGE.search(heading):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


# ============================================================================
# STEP 2: document + section lookup
# ============================================================================
def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# STEP 3: table-row flattening (fixes the page-break table-split issue)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    """Flatten all table-objects in a section into one row list, forward-fill
    blank continuation labels, group into paragraph segments. No data is dropped
    -- every row's text is preserved inside some segment."""
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))

    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))

    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])

    return [(label, " ".join(texts).strip()) for label, texts in segments]


# ============================================================================
# STEP 4: main alignment loop
# ============================================================================
def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(
        set(en_docs) | set(pa_docs),
        key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]),
    )

    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))

        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = {s["section_id"]: s for s in (en_doc["sections"] if en_doc else [])}
        pa_sec_map = {s["section_id"]: s for s in (pa_doc["sections"] if pa_doc else [])}

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            en_present = en_sec is not None
            pa_present = pa_sec is not None
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            # A section containing the Agromet weather-forecast table (Wind, Temperature,
            # Rainfall, etc.) has that table -- and ONLY that table -- fully compiled
            # separately in the Agromet_Weather_Tables sheet, so its heading row and
            # table_row entries are skipped here to avoid duplication. Its other content
            # (narrative text_blocks like "Date: ..." or the forecast summary) still
            # belongs on this sheet and is kept.
            section_has_weather_table = any(
                is_weather_table(t)
                for sec in (en_sec, pa_sec) if sec
                for t in sec.get("tables", [])
            )

            def reason_for(en_text, pa_text):
                """Explain WHY a row has a blank side, so it's auditable, not just missing."""
                en_blank = not (en_text and en_text.strip())
                pa_blank = not (pa_text and pa_text.strip())
                if not en_blank and not pa_blank:
                    return "Aligned (both sides present)"
                if en_blank and not pa_blank:
                    if not en_present:
                        return "Whole section missing in English JSON for this date"
                    return "Content gap: this item missing in English JSON (section otherwise present)"
                if pa_blank and not en_blank:
                    if not pa_present:
                        return "Whole section missing in Punjabi JSON for this date"
                    return "Content gap: this item missing in Punjabi JSON (section otherwise present)"
                return "Both sides blank"

            if not section_has_weather_table:
                records.append({**base, "Content_Type": "heading",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": heading_en, "Punjabi_Text": heading_pa,
                                 "Missing_Reason": reason_for(heading_en, heading_pa)})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else "",
                                 "Missing_Reason": reason_for(en_b["text"] if en_b else "", pa_b["text"] if pa_b else "")})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                en_t = en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else "")
                pa_t = pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_t, "Punjabi_Text": pa_t,
                                 "Missing_Reason": reason_for(en_t, pa_t)})

            if not section_has_weather_table:
                en_segments = flatten_and_group_tables(en_sec)
                pa_segments = flatten_and_group_tables(pa_sec)
                for i in range(max(len(en_segments), len(pa_segments))):
                    en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                    pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                    records.append({**base, "Content_Type": "table_row",
                                     "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                     "English_Text": en_text, "Punjabi_Text": pa_text,
                                     "Missing_Reason": reason_for(en_text, pa_text)})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text", "Missing_Reason",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])

    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(
        by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable"
    ).drop(columns="_m").reset_index(drop=True)

    return df, missing_df


def build_summary(df):
    """Aggregate missing-content counts by Section_ID and by District, so the
    scale and pattern of gaps is immediately visible and explainable."""
    total = len(df)
    reason_counts = df["Missing_Reason"].value_counts().rename_axis("Missing_Reason").reset_index(name="Row_Count")
    reason_counts["Percent_of_Total"] = (reason_counts["Row_Count"] / total * 100).round(2)

    by_section = (
        df[df["Missing_Reason"] != "Aligned (both sides present)"]
        .groupby(["Section_ID", "Missing_Reason"]).size()
        .reset_index(name="Row_Count")
        .sort_values("Row_Count", ascending=False)
    )

    return reason_counts, by_section


def load_all_docs(root):
    """Return {(district, month, date): parsed_json_dict} for every document under root."""
    doc_paths = list_docs(root)
    return {key: load_json(path) for key, path in doc_paths.items()}


DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def is_weather_table(table):
    """The Agromet Advisory weather-forecast table always has 'Parameter' as the
    first column and actual calendar dates as the remaining column headers --
    detect it structurally (language-independent) rather than by section_id."""
    headers = table.get("headers", [])
    if len(headers) < 2:
        return False
    return all(DATE_PATTERN.match(h) for h in headers[1:])

def extract_weather_rows(doc):
    """Scan every section of a document for the weather table and return
    [(parameter_label, [(date, value), ...]), ...] in original row order."""
    if not doc:
        return []
    rows_out = []
    for sec in doc.get("sections", []):
        for t in sec.get("tables", []):
            if not is_weather_table(t):
                continue
            headers = t["headers"]
            param_header = headers[0]
            date_headers = headers[1:]
            for row in t.get("rows", []):
                param = row.get(param_header, "")
                values = [(d, row.get(d, "")) for d in date_headers]
                rows_out.append((param, values))
    return rows_out

def build_weather_df(en_docs_raw, pa_docs_raw, all_keys):
    """Compile the Agromet weather-parameter tables (Wind, Temperature, Rainfall,
    Humidity, Cloud Cover, Warning, etc.) from every district/date into one
    uniform, side-by-side EN/PA sheet. Rows matched positionally by parameter
    order, which is stable across all bulletins (verified against source data)."""
    weather_records = []
    for (district, month, date) in all_keys:
        en_doc = en_docs_raw.get((district, month, date))
        pa_doc = pa_docs_raw.get((district, month, date))
        en_rows = extract_weather_rows(en_doc)
        pa_rows = extract_weather_rows(pa_doc)
        max_rows = max(len(en_rows), len(pa_rows))
        for i in range(max_rows):
            en_param, en_vals = en_rows[i] if i < len(en_rows) else ("", [])
            pa_param, pa_vals = pa_rows[i] if i < len(pa_rows) else ("", [])
            rec = {
                "District": district, "Month": month, "Date": date,
                "Parameter_ENG": en_param, "Parameter_PUN": pa_param,
            }
            missing_bits = []
            if not en_param and pa_param:
                missing_bits.append("Row missing on English side")
            if not pa_param and en_param:
                missing_bits.append("Row missing on Punjabi side")
            for day_idx in range(5):
                en_date, en_val = en_vals[day_idx] if day_idx < len(en_vals) else ("", "")
                pa_date, pa_val = pa_vals[day_idx] if day_idx < len(pa_vals) else ("", "")
                forecast_date = en_date or pa_date
                rec[f"Forecast_Date_Day{day_idx+1}"] = forecast_date
                rec[f"EN_Value_Day{day_idx+1}"] = en_val
                rec[f"PA_Value_Day{day_idx+1}"] = pa_val
                if en_val == "" and pa_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on English side")
                elif pa_val == "" and en_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on Punjabi side")
            rec["Missing_Reason"] = "; ".join(missing_bits) if missing_bits else "Aligned (both sides present)"
            weather_records.append(rec)

    cols = ["District", "Month", "Date", "Parameter_ENG", "Parameter_PUN"]
    for d in range(1, 6):
        cols += [f"Forecast_Date_Day{d}", f"EN_Value_Day{d}", f"PA_Value_Day{d}"]
    cols.append("Missing_Reason")
    wdf = pd.DataFrame(weather_records, columns=cols)
    wdf["_m"] = wdf["Month"].map(MONTH_ORDER).fillna(99)
    wdf = wdf.sort_values(by=["District", "_m", "Date"], kind="stable").drop(columns="_m").reset_index(drop=True)
    return wdf






# ============================================================================
# STEP 5: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, weather_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 11,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    # ---- Agromet_Weather_Tables sheet is written below ----

    # ---- Agromet_Weather_Tables: the weather-forecast tables (Wind, Temperature,
    # Rainfall, Humidity, Cloud Cover, Warning) from every district, compiled into
    # one clean, uniformly-columned sheet -- separate from the advisory tables ----
    ws_w = wb.create_sheet("Agromet_Weather_Tables")
    w_headers = list(weather_df.columns)
    ws_w.append(w_headers)
    for cell in ws_w[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in weather_df.itertuples(index=False):
        ws_w.append(list(row))
    w_wrap_cols = {get_column_letter(w_headers.index(c) + 1) for c in
                   ("Parameter_ENG", "Parameter_PUN", "Missing_Reason") if c in w_headers}
    for row in ws_w.iter_rows(min_row=2, max_row=ws_w.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in w_wrap_cols \
                else Alignment(vertical="top")
    w_widths = {"District": 16, "Month": 12, "Date": 12, "Parameter_ENG": 22, "Parameter_PUN": 22,
                "Missing_Reason": 45}
    for i, h in enumerate(w_headers, start=1):
        width = w_widths.get(h, 14 if "Value" in h or "Forecast" in h else 16)
        ws_w.column_dimensions[get_column_letter(i)].width = width
    ws_w.freeze_panes = "A2"
    ws_w.auto_filter.ref = ws_w.dimensions

    # ---- summary sheets: quantify and explain every gap, for review/presentation ----
    reason_counts, by_section = build_summary(df)

    ws3 = wb.create_sheet("Missing_Content_Summary")
    ws3.append(["Overall breakdown: why each row's English_Text / Punjabi_Text is blank (if it is)"])
    ws3["A1"].font = Font(name="Calibri", size=12, bold=True)
    ws3.append([])
    ws3.append(list(reason_counts.columns))
    for cell in ws3[ws3.max_row]:
        cell.font = header_font
    for r in reason_counts.itertuples(index=False):
        ws3.append(list(r))
    for row in ws3.iter_rows(min_row=4, max_row=ws3.max_row):
        for cell in row:
            cell.font = data_font
    ws3.column_dimensions["A"].width = 55
    ws3.column_dimensions["B"].width = 14
    ws3.column_dimensions["C"].width = 16

    start_row = ws3.max_row + 3
    ws3.cell(row=start_row, column=1, value="Breakdown by Section_ID (which sections have the most gaps, and why)").font = Font(name="Calibri", size=12, bold=True)
    header_row = start_row + 2
    for i, h in enumerate(list(by_section.columns), start=1):
        c = ws3.cell(row=header_row, column=i, value=h)
        c.font = header_font
    for r_i, r in enumerate(by_section.itertuples(index=False), start=header_row + 1):
        for c_i, v in enumerate(r, start=1):
            ws3.cell(row=r_i, column=c_i, value=v).font = data_font
    ws3.freeze_panes = "A4"

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("Step 1/4: Locating 'structured' folders...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")

    print("Step 2/4: Auto-detecting language of each side...")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print("Step 3/5: Aligning paragraph-wise (section-by-section, table-split-safe)...")
    df, missing_df = build_aligned_data(en_root, pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    print("Step 4/5: Compiling Agromet weather-parameter tables (Wind, Temperature, Rainfall, etc.)...")
    en_docs_raw = load_all_docs(en_root)
    pa_docs_raw = load_all_docs(pa_root)
    all_keys = sorted(set(en_docs_raw) | set(pa_docs_raw),
                       key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]))
    weather_df = build_weather_df(en_docs_raw, pa_docs_raw, all_keys)
    print(f"  -> {len(weather_df)} weather-parameter rows across all districts")

    print("Step 5/5: Writing Excel workbook...")
    write_excel(df, missing_df, weather_df, OUTPUT_XLSX)
    print(f"Done. Saved to: {OUTPUT_XLSX}")

Step 1/4: Locating 'structured' folders...
Step 2/4: Auto-detecting language of each side...
  -> English content found at: /content/pipeline_extracted/side_2/Punjab/structured
  -> Punjabi content found at: /content/pipeline_extracted/side_1/Punjab/structured
Step 3/5: Aligning paragraph-wise (section-by-section, table-split-safe)...
  -> 21480 aligned rows, 22 bulletins missing on one side
Step 4/5: Compiling Agromet weather-parameter tables (Wind, Temperature, Rainfall, etc.)...
  -> 5751 weather-parameter rows across all districts
Step 5/5: Writing Excel workbook...
Done. Saved to: /content/Punjab_Agromet_EN_PA_Aligned_Paragraphwise.xlsx


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install pymupdf --quiet

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 83.5 MB/s eta 0:00:00


In [ ]:
"""
FULL PIPELINE WITH PDF RECOVERY: English-Punjabi Agromet Bulletin Alignment
==========================================================================
This is the combined script. In one run it will:

  STEP A: Unzip your two source JSON zips (or use folders directly) and
          auto-detect which side is English vs Punjabi.
  STEP B: Find every section that's genuinely missing on one language side,
          cross-check it against the matching source PDF's text, and where
          the content is actually found in the PDF, PATCH a new section
          into a COPY of that JSON file (your originals are never touched).
          Every patched section is clearly marked (section_id starts with
          "sec_recovered_...", plus a "_recovered_from_pdf": true flag) so
          it's always distinguishable from genuinely-OCR-extracted content.
  STEP C: Run the full paragraph-wise alignment (with the section-category
          fix that prevents Crop/Livestock/etc. cross-matching, and the
          page-break table-split fix) on the PATCHED data.
  STEP D: Write the final Excel workbook with 4 sheets, including the
          recovered content already merged in -- no manual copy-paste needed.

>>> EDIT ONLY THE CONFIG SECTION BELOW, THEN RUN THE WHOLE SCRIPT. <<<
"""

import json, os, glob, re, zipfile, shutil
from collections import Counter
import pandas as pd
import fitz  # PyMuPDF -- pip install pymupdf
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================================
# CONFIG -- edit these paths only
# ============================================================================
# --- Original source JSON (Option A: zips, or Option B: already-unzipped folders) ---
ZIP_PATH_1 = "/content/Punjab-1.zip"
ZIP_PATH_2 = "/content/Punjab(1).zip"
FOLDER_1 = None
FOLDER_2 = None
EXTRACT_DIR = "/content/pipeline_extracted"

# --- Source PDFs, for recovering missing content ---
PDF_ROOT = "/content/drive/MyDrive/Punjab_Agromet/pdfs"

# --- Where the patched JSON copies get written (must end in 'structured') ---
PATCHED_EN_ROOT = "/content/patched/english/structured"
PATCHED_PA_ROOT = "/content/patched/punjabi/structured"

# --- Outputs ---
RECOVERY_LOG_XLSX = "/content/Recovery_Patch_Log.xlsx"
OUTPUT_XLSX = "/content/Punjab_Agromet_EN_PA_Aligned_FINAL.xlsx"

MIN_CHARS_PER_PAGE_FOR_DIGITAL = 40
MONTH_ORDER = {"September": 1, "October": 2, "November": 3, "December": 4}


# ============================================================================
# SHARED: unzip / language detection / JSON reading
# ============================================================================
def prepare_root(zip_path, folder_path, extract_subdir):
    """Return the path to the 'structured' folder, unzipping first if needed."""
    if zip_path:
        target = os.path.join(EXTRACT_DIR, extract_subdir)
        os.makedirs(target, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target)
        search_root = target
    else:
        search_root = folder_path
    for dirpath, dirnames, _ in os.walk(search_root):
        if os.path.basename(dirpath) == "structured":
            return dirpath
    raise FileNotFoundError(f"Could not find a 'structured' folder under {search_root}")


GURMUKHI_RANGE = re.compile(r"[\u0A00-\u0A7F]")

def detect_language(structured_root, sample_size=5):
    json_files = glob.glob(os.path.join(structured_root, "*", "*", "*.json"))[:sample_size]
    if not json_files:
        raise FileNotFoundError(f"No JSON files found under {structured_root}")
    gurmukhi_hits = 0
    for f in json_files:
        doc = json.load(open(f, encoding="utf-8"))
        for sec in doc.get("sections", [])[:2]:
            if GURMUKHI_RANGE.search(sec.get("heading", "")):
                gurmukhi_hits += 1
            break
    return "punjabi" if gurmukhi_hits > 0 else "english"


def list_docs(root):
    docs = {}
    for district in os.listdir(root):
        dpath = os.path.join(root, district)
        if not os.path.isdir(dpath):
            continue
        for month in os.listdir(dpath):
            mpath = os.path.join(dpath, month)
            if not os.path.isdir(mpath):
                continue
            for f in glob.glob(os.path.join(mpath, "*.json")):
                date = os.path.splitext(os.path.basename(f))[0]
                docs[(district, month, date)] = f
    return docs


def load_json(path):
    if path is None:
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ============================================================================
# SHARED: language-agnostic section classification (fixes the section_id
# renumbering bug -- see chat history for the full explanation and the
# real-data evidence: 10.4% of documents / 291 section slots affected)
# ============================================================================
def classify_heading(heading):
    h = (heading or "").strip()
    if not h:
        return "EMPTY"
    if h == "__preamble__":
        return "PREAMBLE"
    if "Gramin Krishi" in h or "Agricultural University" in h:
        return "BULLETIN_TITLE"
    if "ਖੇਤੀਬਾੜੀ" in h and "ਯੂਨੀਵਰਸਿਟੀ" in h:
        return "BULLETIN_TITLE"
    if re.match(r"^(ਗੁਮੀਣ|ਗ੍ਰਾਮੀਣ|ਗ੍ਰੀਨ)", h) and "ਸੇਵਾ" in h:
        return "BULLETIN_TITLE"
    if h == "Agromet Advisory Bulletin":
        return "WEATHER_TABLE_HEADING"
    if "ਸਲਾਹ ਬੁਲੈਟਿਨ" in h and "ਯੂਨੀਵਰਸਿਟੀ" not in h:
        return "WEATHER_TABLE_HEADING"
    if "SMS Advisory" in h:
        return "SMS_ADVISORY"
    if re.search(r"ਐਸ\s*ਐ[ਮਸ]\s*ਐਸ\s*ਸਲਾਹ", h):
        return "SMS_ADVISORY"
    if "Forecast Summary" in h:
        return "FORECAST_SUMMARY"
    if "ਸੰਖੇਪ" in h:
        return "FORECAST_SUMMARY"
    if "Weather Warnings" in h:
        return "WEATHER_WARNINGS"
    if ("ਚੇਤਾਵਨੀ" in h and "ਚੇਤਾਵਨੀਆਂ" not in h) or "IST" in h or "ਮਾਨਯ" in h:
        return "WEATHER_WARNINGS"
    if "on Agriculture and associated Agromet advisories" in h:
        return "LIKELY_IMPACTS_DETAILED"
    if "ਚੇਤਾਵਨੀਆਂ" in h and "ਕਿਸਾਨ" in h:
        return "LIKELY_IMPACTS_DETAILED"
    if "Likely impacts of weather warnings" in h:
        return "LIKELY_IMPACTS_GENERAL"
    if "ਚੇਤਾਵਨੀਆਂ" in h and "ਸੰਭਾਵਿਤ ਪ੍ਰਭਾਵ" in h:
        return "LIKELY_IMPACTS_GENERAL"
    if "Impact based advisories" in h:
        return "IMPACT_BASED_ADVISORIES"
    if "ਪ੍ਰਭਾਵ ਅਧਾਰਤ ਸਲਾਹਾਂ" in h:
        return "IMPACT_BASED_ADVISORIES"
    if h.startswith("General Advisory"):
        return "GENERAL_ADVISORY"
    if "ਸਧਾਰਨ ਸਲਾਹ" in h:
        return "GENERAL_ADVISORY"
    if "Crop Specific Advisory" in h or "ਫਸਲ" in h:
        return "CROP_ADVISORY"
    if "Horticulture Specific Advisory" in h or "ਬਾਗਬਾਨੀ" in h:
        return "HORTICULTURE_ADVISORY"
    if "Live Stock Specific Advisory" in h or "ਲਾਈਵ ਸਟਾਕ" in h:
        return "LIVESTOCK_ADVISORY"
    if "Poultry Specific Advisory" in h or re.search(r"ਪ[ੋੇ]ਲਟਰੀ", h):
        return "POULTRY_ADVISORY"
    if "Others (Soil" in h or "ਦੂਸਰੇ" in h:
        return "OTHERS_SOIL_ADVISORY"
    return "UNKNOWN"


def build_category_map(doc):
    cat_map = {}
    seen_counts = {}
    for sec in (doc.get("sections", []) if doc else []):
        cat = classify_heading(sec.get("heading", ""))
        seen_counts[cat] = seen_counts.get(cat, 0) + 1
        key = cat if seen_counts[cat] == 1 else f"{cat}#{seen_counts[cat]}"
        cat_map[key] = sec
    return cat_map


# ============================================================================
# STEP B FUNCTIONS: PDF recovery + JSON patching
# ============================================================================
def build_canonical_headings(en_docs, pa_docs):
    en_counts, pa_counts = {}, {}
    for path in en_docs.values():
        doc = load_json(path)
        for sec in (doc.get("sections", []) if doc else []):
            cat = classify_heading(sec.get("heading", ""))
            h = sec.get("heading", "").strip()
            if h:
                en_counts.setdefault(cat, Counter())[h] += 1
    for path in pa_docs.values():
        doc = load_json(path)
        for sec in (doc.get("sections", []) if doc else []):
            cat = classify_heading(sec.get("heading", ""))
            h = sec.get("heading", "").strip()
            if h:
                pa_counts.setdefault(cat, Counter())[h] += 1
    en_canonical = {cat: c.most_common(1)[0][0] for cat, c in en_counts.items()}
    pa_canonical = {cat: c.most_common(1)[0][0] for cat, c in pa_counts.items()}
    return en_canonical, pa_canonical


def find_missing_category_cases(en_docs, pa_docs):
    all_keys = sorted(set(en_docs) | set(pa_docs),
                       key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]))
    cases = []
    for key in all_keys:
        district, month, date = key
        en_doc = load_json(en_docs.get(key))
        pa_doc = load_json(pa_docs.get(key))
        en_cats = {classify_heading(s.get("heading", "")) for s in (en_doc.get("sections", []) if en_doc else [])}
        pa_cats = {classify_heading(s.get("heading", "")) for s in (pa_doc.get("sections", []) if pa_doc else [])}
        for cat in (en_cats - pa_cats):
            if cat in ("EMPTY", "UNKNOWN", "PREAMBLE"):
                continue
            cases.append({"District": district, "Month": month, "Date": date,
                           "Category": cat, "Missing_Side": "Punjabi"})
        for cat in (pa_cats - en_cats):
            if cat in ("EMPTY", "UNKNOWN", "PREAMBLE"):
                continue
            cases.append({"District": district, "Month": month, "Date": date,
                           "Category": cat, "Missing_Side": "English"})
    return cases


_pdf_index_cache = None

def build_pdf_index(pdf_root):
    index = {}
    date_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")
    for path in glob.glob(os.path.join(pdf_root, "**", "*.pdf"), recursive=True):
        fname = os.path.basename(path)
        m = date_pattern.search(fname)
        key_date = m.group(0) if m else None
        index.setdefault(key_date, []).append(path)
    return index


def find_pdf(district, date):
    global _pdf_index_cache
    if _pdf_index_cache is None:
        _pdf_index_cache = build_pdf_index(PDF_ROOT)
    candidates = _pdf_index_cache.get(date, [])
    if not candidates:
        return None, False
    district_norm = district.lower().replace(" ", "")
    for c in candidates:
        if district_norm in c.lower().replace(" ", ""):
            return c, True
    if len(candidates) == 1:
        return candidates[0], False
    return None, False


def extract_pdf_text(pdf_path):
    try:
        doc = fitz.open(pdf_path)
    except Exception:
        return "", True
    texts = [page.get_text() for page in doc]
    n_pages = max(len(doc), 1)
    doc.close()
    full_text = "\n".join(texts)
    avg_chars = len(full_text) / n_pages
    return full_text, avg_chars < MIN_CHARS_PER_PAGE_FOR_DIGITAL


def normalize(s):
    return re.sub(r"\s+", " ", s or "").strip()


def extract_recovered_text(pdf_text, target_heading, all_headings_this_language):
    norm_text = normalize(pdf_text)
    norm_target = normalize(target_heading)
    if not norm_target:
        return None
    start_idx = norm_text.find(norm_target)
    if start_idx == -1:
        return None
    content_start = start_idx + len(norm_target)
    next_boundary = len(norm_text)
    for h in all_headings_this_language:
        h_norm = normalize(h)
        if not h_norm or h_norm == norm_target:
            continue
        idx = norm_text.find(h_norm, content_start)
        if idx != -1 and idx < next_boundary:
            next_boundary = idx
    recovered = norm_text[content_start:next_boundary].strip(" :।-")
    return recovered if recovered else None


def make_recovered_section(existing_sections, heading_text, recovered_text):
    existing_ids = [s.get("section_id", "") for s in existing_sections]
    n = 1
    while f"sec_recovered_{n:03d}" in existing_ids:
        n += 1
    return {
        "section_id": f"sec_recovered_{n:03d}",
        "heading": heading_text,
        "text_blocks": [{"text": recovered_text, "page_no": None}],
        "list_items": [],
        "tables": [],
        "_recovered_from_pdf": True,
    }


def run_patching(en_root, pa_root):
    """STEP B: copy JSON trees, find gaps, cross-check PDFs, patch recovered
    content into the copies. Returns (patched_en_root, patched_pa_root)."""
    print("  Copying JSON trees to patched output locations...")
    if os.path.exists(PATCHED_EN_ROOT):
        shutil.rmtree(PATCHED_EN_ROOT)
    if os.path.exists(PATCHED_PA_ROOT):
        shutil.rmtree(PATCHED_PA_ROOT)
    shutil.copytree(en_root, PATCHED_EN_ROOT)
    shutil.copytree(pa_root, PATCHED_PA_ROOT)

    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    en_docs_patched = list_docs(PATCHED_EN_ROOT)
    pa_docs_patched = list_docs(PATCHED_PA_ROOT)

    print("  Building canonical headings per category/language...")
    en_canonical, pa_canonical = build_canonical_headings(en_docs, pa_docs)
    en_all_headings = list(en_canonical.values())
    pa_all_headings = list(pa_canonical.values())

    print("  Finding genuinely missing-category cases...")
    cases = find_missing_category_cases(en_docs, pa_docs)
    print(f"    -> {len(cases)} flagged cases to check")

    print("  Cross-checking against PDFs and patching recovered content...")
    log_rows = []
    pdf_text_cache = {}
    patched_files_touched = set()

    for case in cases:
        district, month, date, cat, missing_side = (
            case["District"], case["Month"], case["Date"], case["Category"], case["Missing_Side"]
        )
        target_heading = pa_canonical.get(cat, "") if missing_side == "Punjabi" else en_canonical.get(cat, "")
        all_headings = pa_all_headings if missing_side == "Punjabi" else en_all_headings

        pdf_path, district_matched = find_pdf(district, date)
        row = {**case, "Heading_Searched": target_heading, "PDF_Found": pdf_path is not None,
               "PDF_District_Matched": district_matched, "PDF_Path": pdf_path or "",
               "Verdict": "", "Recovered_Text": "", "Patched_File": ""}

        if pdf_path is None:
            row["Verdict"] = "PDF not found -- could not verify"
        else:
            if pdf_path not in pdf_text_cache:
                pdf_text_cache[pdf_path] = extract_pdf_text(pdf_path)
            pdf_text, is_scanned = pdf_text_cache[pdf_path]

            if is_scanned:
                row["Verdict"] = "PDF has no reliable text layer -- needs manual visual check"
            elif not target_heading:
                row["Verdict"] = "No canonical heading known for this category -- needs manual check"
            else:
                recovered = extract_recovered_text(pdf_text, target_heading, all_headings)
                if recovered:
                    key = (district, month, date)
                    target_docs_patched = pa_docs_patched if missing_side == "Punjabi" else en_docs_patched
                    patch_path = target_docs_patched.get(key)
                    if patch_path is None:
                        row["Verdict"] = "RECOVERED but target JSON file not found -- could not patch"
                    else:
                        doc = load_json(patch_path)
                        new_section = make_recovered_section(doc.get("sections", []), target_heading, recovered)
                        doc.setdefault("sections", []).append(new_section)
                        with open(patch_path, "w", encoding="utf-8") as f:
                            json.dump(doc, f, ensure_ascii=False, indent=2)
                        patched_files_touched.add(patch_path)
                        row["Verdict"] = "RECOVERED and PATCHED into JSON"
                        row["Recovered_Text"] = recovered
                        row["Patched_File"] = patch_path
                else:
                    row["Verdict"] = "GENUINELY ABSENT -- heading not found anywhere in PDF text"
        log_rows.append(row)

    print("  Writing recovery/patch log...")
    log_df = pd.DataFrame(log_rows)
    log_df["_m"] = log_df["Month"].map(MONTH_ORDER).fillna(99)
    log_df = log_df.sort_values(by=["District", "_m", "Date", "Category"]).drop(columns="_m")
    with pd.ExcelWriter(RECOVERY_LOG_XLSX, engine="openpyxl") as writer:
        log_df.to_excel(writer, sheet_name="Recovery_Patch_Log", index=False)
        summary = log_df["Verdict"].value_counts().rename_axis("Verdict").reset_index(name="Count")
        summary.to_excel(writer, sheet_name="Verdict_Summary", index=False)

    print(f"    {len(patched_files_touched)} JSON files patched with recovered content")
    print("    Verdict breakdown:", dict(log_df["Verdict"].value_counts()))
    return PATCHED_EN_ROOT, PATCHED_PA_ROOT


# ============================================================================
# STEP C FUNCTIONS: paragraph-wise alignment (table-split-safe, category-matched)
# ============================================================================
def row_label_and_text(row, headers):
    if not headers:
        return "", ""
    label = row.get(headers[0], "")
    rest_headers = headers[1:]
    if len(rest_headers) == 1:
        text = row.get(rest_headers[0], "")
    else:
        parts = [f"{h}: {row.get(h, '')}" for h in rest_headers if row.get(h, "") != ""]
        text = "; ".join(parts)
    return label, text


def flatten_and_group_tables(section):
    if not section:
        return []
    flat_rows = []
    for table in section.get("tables", []):
        headers = table.get("headers", [])
        for row in table.get("rows", []):
            flat_rows.append(row_label_and_text(row, headers))
    effective = []
    last_label = ""
    for label, text in flat_rows:
        if label and label.strip():
            last_label = label.strip()
        effective.append((last_label, text))
    segments = []
    for eff_label, text in effective:
        if segments and segments[-1][0] == eff_label:
            if text:
                segments[-1][1].append(text)
        else:
            segments.append([eff_label, [text] if text else []])
    return [(label, " ".join(texts).strip()) for label, texts in segments]


DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def is_weather_table(table):
    headers = table.get("headers", [])
    if len(headers) < 2:
        return False
    return all(DATE_PATTERN.match(h) for h in headers[1:])


def extract_weather_rows(doc):
    if not doc:
        return []
    rows_out = []
    for sec in doc.get("sections", []):
        for t in sec.get("tables", []):
            if not is_weather_table(t):
                continue
            headers = t["headers"]
            param_header = headers[0]
            date_headers = headers[1:]
            for row in t.get("rows", []):
                param = row.get(param_header, "")
                values = [(d, row.get(d, "")) for d in date_headers]
                rows_out.append((param, values))
    return rows_out


def load_all_docs(root):
    doc_paths = list_docs(root)
    return {key: load_json(path) for key, path in doc_paths.items()}


def build_weather_df(en_docs_raw, pa_docs_raw, all_keys):
    weather_records = []
    for (district, month, date) in all_keys:
        en_doc = en_docs_raw.get((district, month, date))
        pa_doc = pa_docs_raw.get((district, month, date))
        en_rows = extract_weather_rows(en_doc)
        pa_rows = extract_weather_rows(pa_doc)
        max_rows = max(len(en_rows), len(pa_rows))
        for i in range(max_rows):
            en_param, en_vals = en_rows[i] if i < len(en_rows) else ("", [])
            pa_param, pa_vals = pa_rows[i] if i < len(pa_rows) else ("", [])
            rec = {"District": district, "Month": month, "Date": date,
                   "Parameter_ENG": en_param, "Parameter_PUN": pa_param}
            missing_bits = []
            if not en_param and pa_param:
                missing_bits.append("Row missing on English side")
            if not pa_param and en_param:
                missing_bits.append("Row missing on Punjabi side")
            for day_idx in range(5):
                en_date, en_val = en_vals[day_idx] if day_idx < len(en_vals) else ("", "")
                pa_date, pa_val = pa_vals[day_idx] if day_idx < len(pa_vals) else ("", "")
                forecast_date = en_date or pa_date
                rec[f"Forecast_Date_Day{day_idx+1}"] = forecast_date
                rec[f"EN_Value_Day{day_idx+1}"] = en_val
                rec[f"PA_Value_Day{day_idx+1}"] = pa_val
                if en_val == "" and pa_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on English side")
                elif pa_val == "" and en_val != "":
                    missing_bits.append(f"Day{day_idx+1} value missing on Punjabi side")
            rec["Missing_Reason"] = "; ".join(missing_bits) if missing_bits else "Aligned (both sides present)"
            weather_records.append(rec)
    cols = ["District", "Month", "Date", "Parameter_ENG", "Parameter_PUN"]
    for d in range(1, 6):
        cols += [f"Forecast_Date_Day{d}", f"EN_Value_Day{d}", f"PA_Value_Day{d}"]
    cols.append("Missing_Reason")
    wdf = pd.DataFrame(weather_records, columns=cols)
    wdf["_m"] = wdf["Month"].map(MONTH_ORDER).fillna(99)
    wdf = wdf.sort_values(by=["District", "_m", "Date"], kind="stable").drop(columns="_m").reset_index(drop=True)
    return wdf


def build_aligned_data(en_root, pa_root):
    en_docs = list_docs(en_root)
    pa_docs = list_docs(pa_root)
    all_keys = sorted(set(en_docs) | set(pa_docs),
                       key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]))
    records = []
    missing_doc_records = []

    for (district, month, date) in all_keys:
        en_path = en_docs.get((district, month, date))
        pa_path = pa_docs.get((district, month, date))
        if en_path is None or pa_path is None:
            missing_doc_records.append({
                "District": district, "Month": month, "Date": date,
                "English_File_Present": "Yes" if en_path else "No",
                "Punjabi_File_Present": "Yes" if pa_path else "No",
            })

        en_doc = load_json(en_path)
        pa_doc = load_json(pa_path)
        en_sec_map = build_category_map(en_doc)
        pa_sec_map = build_category_map(pa_doc)

        ordered_section_ids = list(en_sec_map.keys())
        for sid in pa_sec_map:
            if sid not in en_sec_map:
                ordered_section_ids.append(sid)

        for sid in ordered_section_ids:
            en_sec = en_sec_map.get(sid)
            pa_sec = pa_sec_map.get(sid)
            en_present = en_sec is not None
            pa_present = pa_sec is not None
            heading_en = en_sec["heading"] if en_sec else ""
            heading_pa = pa_sec["heading"] if pa_sec else ""
            raw_id_en = en_sec.get("section_id", "") if en_sec else ""
            raw_id_pa = pa_sec.get("section_id", "") if pa_sec else ""
            base = {"District": district, "Month": month, "Date": date, "Section_ID": sid,
                    "Raw_Section_ID_ENG": raw_id_en, "Raw_Section_ID_PUN": raw_id_pa,
                    "Section_Heading_ENG": heading_en, "Section_Heading_PUN": heading_pa}

            section_has_weather_table = any(
                is_weather_table(t)
                for sec in (en_sec, pa_sec) if sec
                for t in sec.get("tables", [])
            )

            def reason_for(en_text, pa_text):
                en_blank = not (en_text and en_text.strip())
                pa_blank = not (pa_text and pa_text.strip())
                if not en_blank and not pa_blank:
                    return "Aligned (both sides present)"
                if en_blank and not pa_blank:
                    if not en_present:
                        return "Whole section missing in English JSON for this date"
                    return "Content gap: this item missing in English JSON (section otherwise present)"
                if pa_blank and not en_blank:
                    if not pa_present:
                        return "Whole section missing in Punjabi JSON for this date"
                    return "Content gap: this item missing in Punjabi JSON (section otherwise present)"
                return "Both sides blank"

            if not section_has_weather_table:
                records.append({**base, "Content_Type": "heading",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": heading_en, "Punjabi_Text": heading_pa,
                                 "Missing_Reason": reason_for(heading_en, heading_pa)})

            en_blocks = en_sec["text_blocks"] if en_sec else []
            pa_blocks = pa_sec["text_blocks"] if pa_sec else []
            for i in range(max(len(en_blocks), len(pa_blocks))):
                en_b = en_blocks[i] if i < len(en_blocks) else None
                pa_b = pa_blocks[i] if i < len(pa_blocks) else None
                page_no = (en_b or pa_b or {}).get("page_no", "")
                records.append({**base, "Content_Type": "text_block",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": page_no,
                                 "English_Text": en_b["text"] if en_b else "",
                                 "Punjabi_Text": pa_b["text"] if pa_b else "",
                                 "Missing_Reason": reason_for(en_b["text"] if en_b else "", pa_b["text"] if pa_b else "")})

            en_items = en_sec["list_items"] if en_sec else []
            pa_items = pa_sec["list_items"] if pa_sec else []
            for i in range(max(len(en_items), len(pa_items))):
                en_i = en_items[i] if i < len(en_items) else None
                pa_i = pa_items[i] if i < len(pa_items) else None
                en_t = en_i if isinstance(en_i, str) else (json.dumps(en_i, ensure_ascii=False) if en_i else "")
                pa_t = pa_i if isinstance(pa_i, str) else (json.dumps(pa_i, ensure_ascii=False) if pa_i else "")
                records.append({**base, "Content_Type": "list_item",
                                 "Row_Label_ENG": "", "Row_Label_PUN": "", "Page_No": "",
                                 "English_Text": en_t, "Punjabi_Text": pa_t,
                                 "Missing_Reason": reason_for(en_t, pa_t)})

            if not section_has_weather_table:
                en_segments = flatten_and_group_tables(en_sec)
                pa_segments = flatten_and_group_tables(pa_sec)
                for i in range(max(len(en_segments), len(pa_segments))):
                    en_label, en_text = en_segments[i] if i < len(en_segments) else ("", "")
                    pa_label, pa_text = pa_segments[i] if i < len(pa_segments) else ("", "")
                    records.append({**base, "Content_Type": "table_row",
                                     "Row_Label_ENG": en_label, "Row_Label_PUN": pa_label, "Page_No": "",
                                     "English_Text": en_text, "Punjabi_Text": pa_text,
                                     "Missing_Reason": reason_for(en_text, pa_text)})

    df = pd.DataFrame(records, columns=[
        "District", "Month", "Date", "Section_ID",
        "Raw_Section_ID_ENG", "Raw_Section_ID_PUN",
        "Section_Heading_ENG", "Section_Heading_PUN",
        "Content_Type", "Row_Label_ENG", "Row_Label_PUN",
        "Page_No", "English_Text", "Punjabi_Text", "Missing_Reason",
    ])
    missing_df = pd.DataFrame(missing_doc_records, columns=[
        "District", "Month", "Date", "English_File_Present", "Punjabi_File_Present",
    ])
    df["_m"] = df["Month"].map(MONTH_ORDER).fillna(99)
    df = df.sort_values(by=["District", "_m", "Date", "Section_ID", "Content_Type"], kind="stable").drop(columns="_m").reset_index(drop=True)
    return df, missing_df


def build_summary(df):
    total = len(df)
    reason_counts = df["Missing_Reason"].value_counts().rename_axis("Missing_Reason").reset_index(name="Row_Count")
    reason_counts["Percent_of_Total"] = (reason_counts["Row_Count"] / total * 100).round(2)
    by_section = (
        df[df["Missing_Reason"] != "Aligned (both sides present)"]
        .groupby(["Section_ID", "Missing_Reason"]).size()
        .reset_index(name="Row_Count")
        .sort_values("Row_Count", ascending=False)
    )
    return reason_counts, by_section


# ============================================================================
# STEP D: write plain, structured Excel (no colors, wrapped text, frozen header)
# ============================================================================
def write_excel(df, missing_df, weather_df, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Punjab_Agromet_EN_PA_Aligned"

    headers = list(df.columns)
    ws.append(headers)
    header_font = Font(name="Calibri", size=11, bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    data_font = Font(name="Calibri", size=11)
    wrap_cols = {get_column_letter(headers.index(c) + 1) for c in
                 ("Section_Heading_ENG", "Section_Heading_PUN", "English_Text", "Punjabi_Text")}

    for row in df.itertuples(index=False):
        ws.append(list(row))
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in wrap_cols \
                else Alignment(vertical="top")

    widths = {"District": 16, "Month": 12, "Date": 12, "Section_ID": 24,
              "Raw_Section_ID_ENG": 14, "Raw_Section_ID_PUN": 14,
              "Section_Heading_ENG": 26, "Section_Heading_PUN": 26, "Content_Type": 14,
              "Row_Label_ENG": 20, "Row_Label_PUN": 20, "Page_No": 9,
              "English_Text": 65, "Punjabi_Text": 65, "Missing_Reason": 45}
    for i, h in enumerate(headers, start=1):
        ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 18)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    ws2 = wb.create_sheet("JSON_Missing_Values")
    m_headers = list(missing_df.columns)
    ws2.append(m_headers)
    for cell in ws2[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for r in missing_df.itertuples(index=False):
        ws2.append(list(r))
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.font = data_font
    for i, h in enumerate(m_headers, start=1):
        ws2.column_dimensions[get_column_letter(i)].width = 20
    ws2.freeze_panes = "A2"
    ws2.auto_filter.ref = ws2.dimensions

    ws_w = wb.create_sheet("Agromet_Weather_Tables")
    w_headers = list(weather_df.columns)
    ws_w.append(w_headers)
    for cell in ws_w[1]:
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row in weather_df.itertuples(index=False):
        ws_w.append(list(row))
    w_wrap_cols = {get_column_letter(w_headers.index(c) + 1) for c in
                   ("Parameter_ENG", "Parameter_PUN", "Missing_Reason") if c in w_headers}
    for row in ws_w.iter_rows(min_row=2, max_row=ws_w.max_row):
        for cell in row:
            cell.font = data_font
            cell.alignment = Alignment(wrap_text=True, vertical="top") if cell.column_letter in w_wrap_cols \
                else Alignment(vertical="top")
    w_widths = {"District": 16, "Month": 12, "Date": 12, "Parameter_ENG": 22, "Parameter_PUN": 22,
                "Missing_Reason": 45}
    for i, h in enumerate(w_headers, start=1):
        width = w_widths.get(h, 14 if "Value" in h or "Forecast" in h else 16)
        ws_w.column_dimensions[get_column_letter(i)].width = width
    ws_w.freeze_panes = "A2"
    ws_w.auto_filter.ref = ws_w.dimensions

    reason_counts, by_section = build_summary(df)
    ws3 = wb.create_sheet("Missing_Content_Summary")
    ws3.append(["Overall breakdown: why each row's English_Text / Punjabi_Text is blank (if it is)"])
    ws3["A1"].font = Font(name="Calibri", size=12, bold=True)
    ws3.append([])
    ws3.append(list(reason_counts.columns))
    for cell in ws3[ws3.max_row]:
        cell.font = header_font
    for r in reason_counts.itertuples(index=False):
        ws3.append(list(r))
    for row in ws3.iter_rows(min_row=4, max_row=ws3.max_row):
        for cell in row:
            cell.font = data_font
    ws3.column_dimensions["A"].width = 55
    ws3.column_dimensions["B"].width = 14
    ws3.column_dimensions["C"].width = 16

    start_row = ws3.max_row + 3
    ws3.cell(row=start_row, column=1, value="Breakdown by Section_ID (which sections have the most gaps, and why)").font = Font(name="Calibri", size=12, bold=True)
    header_row = start_row + 2
    for i, h in enumerate(list(by_section.columns), start=1):
        c = ws3.cell(row=header_row, column=i, value=h)
        c.font = header_font
    for r_i, r in enumerate(by_section.itertuples(index=False), start=header_row + 1):
        for c_i, v in enumerate(r, start=1):
            ws3.cell(row=r_i, column=c_i, value=v).font = data_font
    ws3.freeze_panes = "A4"

    wb.save(path)


# ============================================================================
# RUN THE FULL PIPELINE
# ============================================================================
if __name__ == "__main__":
    print("STEP A: Locating and identifying source JSON (English vs Punjabi)...")
    root_1 = prepare_root(ZIP_PATH_1, FOLDER_1, "side_1")
    root_2 = prepare_root(ZIP_PATH_2, FOLDER_2, "side_2")
    lang_1 = detect_language(root_1)
    lang_2 = detect_language(root_2)
    if lang_1 == lang_2:
        raise ValueError(f"Both sides detected as '{lang_1}' -- check your input paths.")
    en_root = root_1 if lang_1 == "english" else root_2
    pa_root = root_2 if lang_1 == "english" else root_1
    print(f"  -> English content found at: {en_root}")
    print(f"  -> Punjabi content found at: {pa_root}")

    print()
    print("STEP B: Recovering missing content from source PDFs and patching JSON copies...")
    patched_en_root, patched_pa_root = run_patching(en_root, pa_root)
    print(f"  -> Patched data ready at: {patched_en_root} / {patched_pa_root}")

    print()
    print("STEP C: Aligning paragraph-wise on the PATCHED data (table-split-safe, category-matched)...")
    df, missing_df = build_aligned_data(patched_en_root, patched_pa_root)
    print(f"  -> {len(df)} aligned rows, {len(missing_df)} bulletins missing on one side")

    en_docs_raw = load_all_docs(patched_en_root)
    pa_docs_raw = load_all_docs(patched_pa_root)
    all_keys = sorted(set(en_docs_raw) | set(pa_docs_raw),
                       key=lambda k: (k[0], MONTH_ORDER.get(k[1], 99), k[2]))
    weather_df = build_weather_df(en_docs_raw, pa_docs_raw, all_keys)
    print(f"  -> {len(weather_df)} weather-parameter rows across all districts")

    print()
    print("STEP D: Writing final Excel workbook...")
    write_excel(df, missing_df, weather_df, OUTPUT_XLSX)
    print(f"Done. Final aligned workbook (with recovered content merged in): {OUTPUT_XLSX}")
    print(f"Recovery audit log: {RECOVERY_LOG_XLSX}")

STEP A: Locating and identifying source JSON (English vs Punjabi)...
  -> English content found at: /content/pipeline_extracted/side_2/Punjab/structured
  -> Punjabi content found at: /content/pipeline_extracted/side_1/Punjab/structured

STEP B: Recovering missing content from source PDFs and patching JSON copies...
  Copying JSON trees to patched output locations...
  Building canonical headings per category/language...
  Finding genuinely missing-category cases...
    -> 666 flagged cases to check
  Cross-checking against PDFs and patching recovered content...
  Writing recovery/patch log...
    0 JSON files patched with recovered content
    Verdict breakdown: {'PDF not found -- could not verify': np.int64(666)}
  -> Patched data ready at: /content/patched/english/structured / /content/patched/punjabi/structured

STEP C: Aligning paragraph-wise on the PATCHED data (table-split-safe, category-matched)...
  -> 21127 aligned rows, 22 bulletins missing on one side
  -> 5751 weather-para